#### Start by importing the code 

In [ ]:
from onsset import *
import os
from IPython.display import display, Markdown, HTML
%matplotlib inline

# 1. GIS data selection

First, run the cell below to browse to the directory your input CSV file is located at and select the input file. 

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox
from openpyxl import load_workbook
root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)
messagebox.showinfo('OnSSET', 'Open the input file with extracted GIS data')
input_file = filedialog.askopenfilename()

onsseter = SettlementProcessor(input_file)
onsseter.df['IsUrban'] = 0
onsseter.df['Conflict'] = 0
onsseter.df['PerCapitaDemand'] = 0
SETTLEMENTS_CSV = input_file

# 2. Modelling period and target electrification rate

Next, define the modelling period and the electrification rate to be achieved by the end of the analysis. Further down you will also define an intermediate year and target.

In [ ]:
start_year = 2018
end_year = 2030
electrification_rate_target = 1 # E.g. 1 for 100% electrification rate or 0.80 for 80% electrification rate 

#### Electricity demand target level
For the second lever, enter the target tier (level of electricity access) for urban and rural households respectively. This can take a value between "1" (lowest level of electricity access) and "5" (highest level of electricity access) as in ESMAPs Multi-Tier Framework for Measuring Electricity Access (found <a href="https://www.esmap.org/node/55526" target="_blank">here</a>). Alternatively, enter "6" to use a distribution of the tiers across the country based on poverty levels and GDP according to the methodology found <a href="https://www.mdpi.com/1996-1073/12/7/1395" target="_blank">here</a>.   

In [ ]:
urban_target_tier = 3
rural_target_tier = 3

#### Intermediate electrification rate target
The model is set up to run in two steps. Enter the intermediate target year and target electrification rate for that year.

In [ ]:
intermediate_year = 2027
intermediate_electrification_target = 0.75 # E.g. for a target electrification rate of 75%, enter 0.75

#### Grid specifications
This part can be used to impose restrictions or forced extensions of the grid. 

In [ ]:
# Buffer distance (km) from the current grid network for automatic connection to the grid.
auto_intensification = 2

# This is the maximum amount of new households that can be connected to the grid in one year (thousands) per time-step
annual_new_grid_connections_limit = {intermediate_year: 1000,
                                     end_year: 999999999}

# This is the maximum generation capacity that can be added to the grid in one year (MW)
annual_grid_cap_gen_limit = {intermediate_year: 20000,
                             end_year: 999999999}

# 4. Enter country specific data

In addition to the options above the user can customize a large number of variables describing the social - economic - technological environment in the selected country. 

**Note!** Most input values shall represent estimates for the variable valid throughout the modelling period, i.e. **NOT** current values.

### a. Demographics and Social components

In [ ]:
pop_start_year = 12100000      ### Write the population in the base year (e.g. 2018) 
end_year_pop = 16600000         ### Write the population in the end year of the analysis (e.g. 2030)

urban_ratio_start_year = 0.484   ### Write the urban population population ratio in the base year (e.g. 2018)
urban_ratio_end_year = 0.57     ### Write the urban population population ratio in the end year (e.g. 2030)

num_people_per_hh_urban = 3.1     ### Write the number of people per household expected in the end year (e.g. 2030)
num_people_per_hh_rural = 3.6   ### Write the number of people per household expected in the end year (e.g. 2030)

elec_ratio_start_year = 0.42   ### Write the electrification rate in the base year (e.g. 2018)
urban_elec_ratio = 0.67       ### Write urban electrification rate in the base year (e.g. 2018)
rural_elec_ratio = 0.18         ### Write rural electrification rate in the base year (e.g. 2018)

### b. Technology specifications & costs

The cell below contains all the information that is used to calculate the levelized costs for all the technologies, including grid. These default values should be updated to reflect the most accurate values in the country. There are currently 7 potential technologies to include in the model:
* Grid
* PV Mini-grid
* Wind Mini-grid
* Hydro Mini-grid
* Diesel Mini-grid
* PV Stand-alone systems
* Diesel Stand-alone systems

First, decide whether to include diesel technologies or not:

In [ ]:
diesel_techs = 1                      ### 0 = diesel NOT included, 1 = diesel included 

In [ ]:
grid_generation_cost = 0.012           ### This is the grid electricity generation cost (USD/kWh) as expected in the end year of the analysis
grid_power_plants_capital_cost = 2300 ### The cost in USD/kW to for capacity upgrades of the grid
grid_losses = 0.082                     ### The fraction of electricity lost in transmission and distribution (percentage) 
base_to_peak = 0.8                    ### The ratio of base grid demand to peak demand (percentage) 
existing_grid_cost_ratio = 0.1        ### The additional cost per round of electrification (percentage) 

In [ ]:
diesel_price = 0.5                   ### This is the diesel price in USD/liter as expected in the end year of the analysis.

In [ ]:
sa_diesel_capital_cost = 938          ### Stand-alone Diesel capital cost (USD/kW) as expected in the years of the analysis
mg_diesel_capital_cost = 721          ### Mini-grid Diesel capital cost (USD/kW) as expected in the years of the analysis
mg_pv_capital_cost = 2950             ### Mini-grid PV capital cost (USD/kW) as expected in the years of the analysis
mg_wind_capital_cost = 2800           ### Mini-grid Wind capital cost (USD/kW) as expected in the years of the analysis
mg_hydro_capital_cost = 3000          ### Mini-grid Hydro capital cost (USD/kW) as expected in the years of the analysis

In [ ]:
sa_pv_capital_cost_1 = 9620          ### Stand-alone PV capital cost (USD/kW) for household systems under 20 W
sa_pv_capital_cost_2 = 8780          ### Stand-alone PV capital cost (USD/kW) for household systems between 21-50 W
sa_pv_capital_cost_3 = 6380           ### Stand-alone PV capital cost (USD/kW) for household systems between 51-100 W
sa_pv_capital_cost_4 = 4470           ### Stand-alone PV capital cost (USD/kW) for household systems between 101-1000 W
sa_pv_capital_cost_5 = 6950           ### Stand-alone PV capital cost (USD/kW) for household systems over 1 kW

The cells below contain additional technology specifications

In [ ]:
discount_rate = 0.08 # E.g. 0.08 means a discount rate of 8%

# Transmission and distribution costs
hv_line_capacity=69 # kV
hv_line_cost=53000 # USD/km
mv_line_cost = 7000 # USD/kW
mv_line_capacity=50 # kV
mv_line_max_length=50 # km
mv_increase_rate=0.1
max_mv_line_dist = 50 # km
MV_line_amperage_limit = 8  # Ampere (A)
lv_line_capacity=0.24 #kV
lv_line_max_length=0.8 # km
lv_line_cost=4250 # USD/km
service_Transf_type=50  # kVa
service_Transf_cost=4250  # $/unit
max_nodes_per_serv_trans=300  # maximum number of nodes served by each service transformer
hv_lv_transformer_cost=25000 # USD/unit
hv_mv_transformer_cost=25000 # USD/unit
mv_lv_transformer_cost=10000 # USD/unit
mv_mv_transformer_cost=10000 # USD/unit


# Centralized grid costs
grid_calc = Technology(om_of_td_lines=0.1,
                        distribution_losses=grid_losses,
                        connection_cost_per_hh=150,
                        base_to_peak_load_ratio=base_to_peak,
                        capacity_factor=1,
                        tech_life=30,
                        grid_capacity_investment=grid_power_plants_capital_cost,
                        grid_price=grid_generation_cost)

# Mini-grid hydro costs
mg_hydro_calc = Technology(om_of_td_lines=0.03,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capacity_factor=0.5,
                            tech_life=30,
                            capital_cost={float("inf"): mg_hydro_capital_cost},
                            om_costs=0.02,
                            )

# Mini-grid wind costs
mg_wind_calc = Technology(om_of_td_lines=0.03,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capital_cost={float("inf"): mg_wind_capital_cost},
                            om_costs=0.02,
                            tech_life=20,
                            )

# Mini-grid PV costs
mg_pv_calc = Technology(om_of_td_lines=0.03,
                        distribution_losses=0.05,
                        connection_cost_per_hh=100,
                        base_to_peak_load_ratio=0.85,
                        tech_life=20,
                        om_costs=0.02,
                        capital_cost={float("inf"): mg_pv_capital_cost}                        
                        )

# Stand-alone PV costs
sa_pv_calc = Technology(base_to_peak_load_ratio=0.9,
                        tech_life=15,
                        om_costs=0.02,
                        capital_cost={float("inf"): sa_pv_capital_cost_5,
                                      1: sa_pv_capital_cost_4,
                                      0.100: sa_pv_capital_cost_3,
                                      0.050: sa_pv_capital_cost_2,
                                      0.020: sa_pv_capital_cost_1},
                        standalone=True
                        )

# Mini-grid diesel costs
mg_diesel_calc = Technology(om_of_td_lines=0.02,
                            distribution_losses=0.05,
                            connection_cost_per_hh=100,
                            base_to_peak_load_ratio=0.85,
                            capacity_factor=0.7,
                            tech_life=15,
                            om_costs=0.1,
                            capital_cost={float("inf"): mg_diesel_capital_cost},
                            mini_grid=True)



# Stand-alone diesel costs
sa_diesel_calc = Technology(base_to_peak_load_ratio=0.9,
                            capacity_factor=0.5,
                            tech_life=10,
                            om_costs=0.1,
                            capital_cost={float("inf"): sa_diesel_capital_cost},
                            standalone=True)

Technology.set_default_values(base_year=start_year, start_year=start_year, end_year=end_year, discount_rate=discount_rate,
                             hv_line_type=hv_line_capacity, hv_line_cost=hv_line_cost, mv_line_type=mv_line_capacity,
                             mv_line_amperage_limit=MV_line_amperage_limit, mv_line_cost=mv_line_cost, lv_line_type=lv_line_capacity,
                             lv_line_cost=lv_line_cost, lv_line_max_length=lv_line_max_length, 
                             service_transf_type=service_Transf_type, service_transf_cost = service_Transf_cost,
                             max_nodes_per_serv_trans=max_nodes_per_serv_trans, mv_lv_sub_station_cost=mv_lv_transformer_cost,
                             mv_mv_sub_station_cost=mv_mv_transformer_cost, hv_lv_sub_station_cost=hv_lv_transformer_cost,
                             hv_mv_sub_station_cost=hv_mv_transformer_cost)

# 5. GIS data import and processing

OnSSET is a GIS based tool and its proper function depends heavily on the diligent preparation and calibration of the necessary geospatial data. Documentation on GIS processing in regards to OnSSET can be found <a href="http://onsset-manual.readthedocs.io/en/latest/data_acquisition.html" target="_blank">here</a>. The following cell reads the CSV-file containing the extracted GIS data for the country chosen in the previous section, and displays a snap-shot of some of the data.

In [ ]:
try:
    onsseter.df.reset_index(inplace=True)
except ValueError:
    pass

# -- Added dummy columns to prevent error -- #
onsseter.df['Elevation'] = 0
onsseter.df['Slope'] = 0
onsseter.df['LandCover'] = 'Other'
onsseter.df['ElectrificationOrder'] = 1
onsseter.df['ResidentialDemandTier1'] = 1
onsseter.df['ResidentialDemandTier2'] = 1
onsseter.df['ResidentialDemandTier3'] = 1
onsseter.df['ResidentialDemandTier4'] = 1
onsseter.df['ResidentialDemandTier5'] = 1

yearsofanalysis = [intermediate_year, end_year]

onsseter.condition_df()
onsseter.df[SET_GRID_PENALTY] = onsseter.grid_penalties(onsseter.df)
onsseter.df[SET_WINDCF] = onsseter.calc_wind_cfs(onsseter.df[SET_WINDVEL])
pop_modelled, urban_modelled = onsseter.calibrate_current_pop_and_urban(pop_start_year, urban_ratio_start_year)
onsseter.project_pop_and_urban(end_year_pop, urban_ratio_end_year, start_year, yearsofanalysis)

eleclimits = {intermediate_year: intermediate_electrification_target, end_year: electrification_rate_target}
time_steps = {intermediate_year: intermediate_year-start_year, end_year: end_year-intermediate_year}

display(Markdown('#### The csv file has been imported correctly. Here is a preview:'))
display(onsseter.df[['Country','Pop','NightLights','TravelHours','GHI','WindVel','Hydropower','HydropowerDist']].sample(7))

# NEW
print("Rows in settlements file:", len(onsseter.df))
print("Columns available:", len(onsseter.df.columns))

In [ ]:
for year in yearsofanalysis:
    mg_diesel_cost = {'diesel_price': diesel_price,
                      'efficiency': 0.33,
                      'diesel_truck_consumption': 33.7,
                      'diesel_truck_volume': 15000}

    sa_diesel_cost = {'diesel_price': diesel_price,
                      'efficiency': 0.28,
                      'diesel_truck_consumption': 14,
                      'diesel_truck_volume': 300}

    try:
        onsseter.diesel_cost_columns(sa_diesel_cost, mg_diesel_cost, year)
    except ValueError:
        print('To update the diesel cost, please re-run the notebook from the first cell')
        break

#### Calibration of currently electrified settlements

The model calibrates which settlements are likely to be electrified in the start year, to match the national statistical values defined above. A settlement is considered to be electrified if it meets all of the following conditions:
- Has more night-time lights than the defined threshold (this is set to 0 by default)
- Is closer to the existing grid network than the distance limit
- Has more population than the threshold

First, define the threshold limits. Then run the calibration and check if the results seem okay. Else, redefine these thresholds and run again.

In [ ]:
min_night_lights = 0    ### 0 Indicates no night light, while any number above refers to the night-lights intensity
min_pop = 100      ### Settlement population above which we can assume that it could be electrified

max_service_transformer_distance = 2    ### Distance in km from the existing grid network below which we can assume a settlement could be electrified
max_mv_line_distance = 3
max_hv_line_distance = 25

elec_calibration_results = onsseter.calibrate_elec_current(elec_ratio_start_year, urban_elec_ratio, rural_elec_ratio, 
                                                           start_year, min_night_lights=min_night_lights, min_pop=min_pop, 
                                                           max_transformer_dist=max_service_transformer_distance, 
                                                           max_mv_dist=max_mv_line_distance, max_hv_dist=max_hv_line_distance,
                                                           buffer=True)

The figure below show the results of the calibration. Settlements in **blue** are considered to be (at least partly) electrified already in the start year of the analysis, while settlements in **yellow** are yet to be electrified. Re-running the calibration step with different intial values may change the map below.

In [ ]:
from matplotlib import pyplot as plt
colors = ['#73B2FF','#EDD100','#EDA800','#1F6600','#98E600','#70A800','#1FA800']
plt.figure(figsize=(9,9))
plt.plot(onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==0, SET_X_DEG], onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==0, SET_Y_DEG], 'y,')
plt.plot(onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==1, SET_X_DEG], onsseter.df.loc[onsseter.df[SET_ELEC_CURRENT]==1, SET_Y_DEG], 'b,')
if onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min() > onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min():
    plt.xlim(onsseter.df[SET_X_DEG].min() - 1, onsseter.df[SET_X_DEG].max() + 1)
    plt.ylim((onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 - 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) - 1, (onsseter.df[SET_Y_DEG].min()+onsseter.df[SET_Y_DEG].max())/2 + 0.5*abs(onsseter.df[SET_X_DEG].max() - onsseter.df[SET_X_DEG].min()) + 1)
else:
    plt.xlim((onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 - 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) - 1, (onsseter.df[SET_X_DEG].min()+onsseter.df[SET_X_DEG].max())/2 + 0.5*abs(onsseter.df[SET_Y_DEG].max() - onsseter.df[SET_Y_DEG].min()) + 1)
    plt.ylim(onsseter.df[SET_Y_DEG].min() -1, onsseter.df[SET_Y_DEG].max() +1)

## 6. Define the demand

This piece of code defines the target electricity demand in the region/country. Residential electricity demand is defined as kWh/household/year, while all other demands are defined as kWh/capita/year. Note that at the moment, all productive uses demands are set to 0 by default.

In [ ]:
# Define the annual household electricity targets to choose from
tier_1 = 38.7  # 38.7 refers to kWh/household/year. 
tier_2 = 219
tier_3 = 803
tier_4 = 2117
tier_5 = 2993

onsseter.prepare_wtf_tier_columns(num_people_per_hh_rural, num_people_per_hh_urban, tier_1, tier_2, tier_3, tier_4, tier_5)

In [ ]:
onsseter.df[SET_EDU_DEMAND] = 0           # Demand for educational facilities (kWh/capita/year)
onsseter.df[SET_HEALTH_DEMAND] = 0        # Demand for health facilities (kWh/capita/year)
onsseter.df[SET_COMMERCIAL_DEMAND] = 0    # Demand for commercial activities (kWh/capita/year)
onsseter.df[SET_AGRI_DEMAND] = 0          # Demand for agricultural activities (kWh/capita/year)
productive_demand = 0 # 1 if productive demand is defined and should be included, else 0

## 7a. Climate-risk settings and technology-hazard typology

In [ ]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import xarray as xr
from scipy.spatial import cKDTree
from scipy.interpolate import RegularGridInterpolator

# Defines the hazards, thresholds, paths, risk typology, and scenario labels used across all climate-risk methods in Section 7


# Hazards to include

ACTIVE_HAZARDS = ["flood", "fire", "heat", "cold", "high_precip", "drought", "landslide", "wind"]


# Hazard thresholds
# Exposure = severity meets or exceeds threshold

HAZARD_THRESHOLDS = {
    "flood": 0.5,          # m
    "wind": 17.0,          # m/s
    "fire": 38.0,          # FWI
    "heat": 38.0,          # °C
    "cold": 0.0,           # °C
    "high_precip": 1800.0, # max monthly precipitation
    "drought": -1.8        # SPI-3 threshold
}


# Hazard paths (local paths for now)

CLIMATE_FOLDER_PATHS = {
    "pr": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/chelsa_data/pr",
    "tasmax": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/chelsa_data/tasmax",
    "tasmin": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/chelsa_data/tasmin"
}

HAZARD_PATHS = {
    "flood_coastal": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/inuncoast_rcp4p5_wtsub_2030_rp0010_0.tif",
    "flood_river": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/inunriver_rcp4p5_0000HadGEM2-ES_2030_rp00010.tif",
    "fire": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/fwixd_ann_HadGEM3-GC31-LL_ssp245_r1i1p1f3_g025.nc",
    "wind_si": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/STORM_FIXED_RETURN_PERIODS_SI_50_YR_RP.tif",
    "wind_ni": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/STORM_FIXED_RETURN_PERIODS_NI_50_YR_RP.tif",
    "wind_na": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/STORM_FIXED_RETURN_PERIODS_NA_50_YR_RP.tif",
    "landslide": "/Users/adshead/Documents/ACCLIMATE/energy-resilience-ssa/LS_RF_Mean_1980-2018_COG.tif"
}

# OnSSET technology names and final codes

TECH_CODES = {
    "Grid": 1,
    "SA_Diesel": 2,
    "SA_PV": 3,
    "MG_Diesel": 4,
    "MG_PV": 5,
    "MG_Wind": 6,
    "MG_Hydro": 7
}

# Hazard-risk typology at subtype level
# 0 = no relevant risk interaction identified
# 1 = indirect or context-dependent (site conditions, surrounding infrastructure, supply chains, user practices, proximity to affected areas) but no direct damage or predictable performance loss to core technology
# 2 = reduced performance or operational disruption (not complete asset failure)
# 3 = direct damage or system failure

# Rule for exclusion method: if settlement is exposed to hazard AND risk > 0, then that technology is excluded there.

TECH_HAZARD_RISK = {
    "Grid": {
        "flood": 3,
        "drought": 2,
        "heat": 2,
        "cold": 3,
        "high_precip": 2,
        "fire": 3,
        "landslide": 3,
        "wind": 3
    },
    "SA_Diesel": {
        "flood": 3,
        "drought": 0,
        "heat": 0,
        "cold": 0,
        "high_precip": 1,
        "fire": 2,
        "landslide": 1,
        "wind": 0
    },
    "SA_PV": {
        "flood": 3,
        "drought": 3,
        "heat": 2,
        "cold": 2,
        "high_precip": 1,
        "fire": 2,
        "landslide": 3,
        "wind": 1
    },
    "MG_Diesel": {
        "flood": 2,
        "drought": 0,
        "heat": 3,
        "cold": 0,
        "high_precip": 1,
        "fire": 2,
        "landslide": 3,
        "wind": 0
    },
    "MG_PV": {
        "flood": 2,
        "drought": 3,
        "heat": 2,
        "cold": 2,
        "high_precip": 2,
        "fire": 2,
        "landslide": 3,
        "wind": 2
    },
    "MG_Wind": {
        "flood": 3,
        "drought": 0,
        "heat": 2,
        "cold": 2,
        "high_precip": 2,
        "fire": 3,
        "landslide": 3,
        "wind": 2
    },
    "MG_Hydro": {
        "flood": 3,
        "drought": 2,
        "heat": 2,
        "cold": 2,
        "high_precip": 2,
        "fire": 3,
        "landslide": 3,
        "wind": 2
    }
}

# Sampled hazard severity columns

HAZARD_VALUE_COLUMNS = {
    "flood": "hazard_flood",
    "wind": "hazard_wind",
    "fire": "hazard_fire",
    "heat": "hazard_heat",
    "cold": "hazard_cold",
    "high_precip": "hazard_high_precip",
    "drought": "hazard_drought_spi3",
    "landslide": "hazard_landslide"
}

# True if severity >= threshold

HAZARD_EXPOSURE_COLUMNS = {
    "flood": "exposed_flood",
    "wind": "exposed_wind",
    "fire": "exposed_fire",
    "heat": "exposed_heat",
    "cold": "exposed_cold",
    "high_precip": "exposed_high_precip",
    "drought": "exposed_drought",
    "landslide": "exposed_landslide"
}


# Scenario labels

BASELINE_SCENARIO = "baseline"
CLIMATE_SCENARIO = "climate_exclusion"
COST_ADJUSTMENT_SCENARIO = "climate_cost"
CAPEX_OPEX_ADJUSTMENT_SCENARIO = "climate_cost_v2"
LIFECYCLE_ADJUSTMENT_SCENARIO = "climate_cost_v3"
PERFORMANCE_ADJUSTMENT_SCENARIO = "climate_performance"

print("Climate-risk configuration loaded.")
print("Scenario labels:")
print("  baseline =", BASELINE_SCENARIO)
print("  exclusion =", CLIMATE_SCENARIO)
print("  cost v1 =", COST_ADJUSTMENT_SCENARIO)
print("  cost v2 =", CAPEX_OPEX_ADJUSTMENT_SCENARIO)
print("  cost v3 =", LIFECYCLE_ADJUSTMENT_SCENARIO)
print("  performance =", PERFORMANCE_ADJUSTMENT_SCENARIO)
print("Active hazards:", ACTIVE_HAZARDS)
print("Hazard thresholds:", HAZARD_THRESHOLDS)


# Risk score needed to trigger exclusion

#   3 = exclude only high-risk interactions
#   2 = exclude moderate + high
#   1 = exclude any non-zero interaction

EXCLUSION_RISK_THRESHOLD = 3

## 7b. Sample hazard layers and create exposure flags

In [ ]:
from scipy.stats import gamma, norm
import os


# Helper functions

def sample_raster(gdf_points, raster_path):
    with rasterio.open(raster_path) as src:
        pts = gdf_points.to_crs(src.crs)
        coords = [(geom.x, geom.y) for geom in pts.geometry]
        vals = np.array([v[0] for v in src.sample(coords)])

        if src.nodata is not None:
            vals = np.where(vals == src.nodata, np.nan, vals)

    return vals


def sample_fire_risk(gdf_points, fire_nc_path, year_index=15):
    # Open dataset
    ds = xr.open_dataset(fire_nc_path)

    # Get coordinate names
    if "lat" in ds.coords:
        lats = ds["lat"].values
    else:
        lats = ds["latitude"].values

    if "lon" in ds.coords:
        lons = ds["lon"].values
    else:
        lons = ds["longitude"].values

    # Extract fire weather index for selected time slice
    fwi = ds["fwixd"].isel(time=year_index).values

    # Optional longitude adjustment if dataset uses 0-360
    point_lons = gdf_points.geometry.x.values.copy()
    if lons.max() > 180:
        point_lons = np.where(point_lons < 0, point_lons + 360, point_lons)

    # Ensure shape matches interpolator expectations
    if fwi.shape != (len(lats), len(lons)):
        raise ValueError(
            f"Shape mismatch: fwi.shape={fwi.shape}, expected ({len(lats)}, {len(lons)})"
        )

    # Build bilinear interpolator
    interp = RegularGridInterpolator(
        (lats, lons),
        fwi,
        bounds_error=False,
        fill_value=np.nan
    )

    # Settlement points in (lat, lon) order
    points = np.vstack([
        gdf_points.geometry.y.values,
        point_lons
    ]).T

    fire_vals = interp(points)

    return fire_vals

MONTHS = [str(m).zfill(2) for m in range(1, 13)]

def stack_and_sample_monthly(gdf_points, folder_path, variable):
    monthly_values = []
    for m in MONTHS:
        filename = f"CHELSA_gfdl-esm4_r1i1p1f1_w5e5_ssp370_{variable}_{m}_2011_2040_norm.tif"
        path = os.path.join(folder_path, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing file: {path}")
        vals = sample_raster(gdf_points, path)
        monthly_values.append(vals)
    return np.array(monthly_values)


def rolling_3month(series):
    out = []
    for start in range(12):
        idx = [(start + i) % 12 for i in range(3)]
        out.append(series[idx].sum())
    return np.array(out)


def compute_spi3(baseline_monthly, future_monthly):
    baseline_3m = rolling_3month(baseline_monthly)
    future_3m = rolling_3month(future_monthly)

    baseline_3m = baseline_3m[baseline_3m > 0]
    if len(baseline_3m) < 5:
        return np.nan

    shape, loc, scale = gamma.fit(baseline_3m, floc=0)
    fut_val = np.nanmin(future_3m)
    prob = gamma.cdf(fut_val, shape, loc=loc, scale=scale)
    spi = norm.ppf(prob)
    return spi


# Ensure GeoDataFrame

if not isinstance(onsseter.df, gpd.GeoDataFrame):
    onsseter.df["geometry"] = gpd.points_from_xy(onsseter.df["X_deg"], onsseter.df["Y_deg"])
    onsseter.df = gpd.GeoDataFrame(onsseter.df, geometry="geometry", crs="EPSG:4326")



# Flood

if "flood" in ACTIVE_HAZARDS:
    onsseter.df["hazard_flood_coastal"] = sample_raster(onsseter.df, HAZARD_PATHS["flood_coastal"])
    onsseter.df["hazard_flood_river"] = sample_raster(onsseter.df, HAZARD_PATHS["flood_river"])
    onsseter.df["hazard_flood"] = onsseter.df[["hazard_flood_coastal", "hazard_flood_river"]].max(axis=1)
    onsseter.df["exposed_flood"] = onsseter.df["hazard_flood"] >= HAZARD_THRESHOLDS["flood"]
else:
    onsseter.df["hazard_flood"] = np.nan
    onsseter.df["exposed_flood"] = False



# Wind

if "wind" in ACTIVE_HAZARDS:
    onsseter.df["hazard_wind_si"] = sample_raster(onsseter.df, HAZARD_PATHS["wind_si"])
    onsseter.df["hazard_wind_ni"] = sample_raster(onsseter.df, HAZARD_PATHS["wind_ni"])
    onsseter.df["hazard_wind_na"] = sample_raster(onsseter.df, HAZARD_PATHS["wind_na"])
    onsseter.df["hazard_wind"] = onsseter.df[["hazard_wind_si", "hazard_wind_ni", "hazard_wind_na"]].max(axis=1)
    onsseter.df["exposed_wind"] = onsseter.df["hazard_wind"] >= HAZARD_THRESHOLDS["wind"]
else:
    onsseter.df["hazard_wind"] = np.nan
    onsseter.df["exposed_wind"] = False



# Fire

if "fire" in ACTIVE_HAZARDS:
    onsseter.df["hazard_fire"] = sample_fire_risk(onsseter.df, HAZARD_PATHS["fire"])
    onsseter.df["exposed_fire"] = onsseter.df["hazard_fire"] >= HAZARD_THRESHOLDS["fire"]
else:
    onsseter.df["hazard_fire"] = np.nan
    onsseter.df["exposed_fire"] = False


# Heat, Cold, High Precip, Drought

if any(h in ACTIVE_HAZARDS for h in ["heat", "cold", "high_precip", "drought"]):
    print("Sampling monthly climate rasters...")

    pr_monthly = None
    tasmax_monthly = None
    tasmin_monthly = None

    if any(h in ACTIVE_HAZARDS for h in ["high_precip", "drought"]):
        print("  Sampling precipitation rasters...")
        pr_monthly = stack_and_sample_monthly(
            onsseter.df,
            CLIMATE_FOLDER_PATHS["pr"],
            "pr"
        )

    if "heat" in ACTIVE_HAZARDS:
        print("  Sampling tasmax rasters...")
        tasmax_monthly = stack_and_sample_monthly(
            onsseter.df,
            CLIMATE_FOLDER_PATHS["tasmax"],
            "tasmax"
        )
        tasmax_monthly = (tasmax_monthly / 10) - 273.15

        onsseter.df["hazard_heat"] = np.nanmax(tasmax_monthly, axis=0)
        onsseter.df["exposed_heat"] = onsseter.df["hazard_heat"] > HAZARD_THRESHOLDS["heat"]
    else:
        onsseter.df["hazard_heat"] = np.nan
        onsseter.df["exposed_heat"] = False

    if "cold" in ACTIVE_HAZARDS:
        print("  Sampling tasmin rasters...")
        tasmin_monthly = stack_and_sample_monthly(
            onsseter.df,
            CLIMATE_FOLDER_PATHS["tasmin"],
            "tasmin"
        )
        tasmin_monthly = (tasmin_monthly / 10) - 273.15

        onsseter.df["hazard_cold"] = np.nanmin(tasmin_monthly, axis=0)
        onsseter.df["exposed_cold"] = onsseter.df["hazard_cold"] < HAZARD_THRESHOLDS["cold"]
    else:
        onsseter.df["hazard_cold"] = np.nan
        onsseter.df["exposed_cold"] = False

    if "high_precip" in ACTIVE_HAZARDS and pr_monthly is not None:
        onsseter.df["hazard_high_precip"] = np.nanmax(pr_monthly, axis=0)
        onsseter.df["exposed_high_precip"] = (
            onsseter.df["hazard_high_precip"] > HAZARD_THRESHOLDS["high_precip"]
        )
    else:
        onsseter.df["hazard_high_precip"] = np.nan
        onsseter.df["exposed_high_precip"] = False

    if "drought" in ACTIVE_HAZARDS and pr_monthly is not None:
        print("  Computing SPI-3 drought...")

        baseline_monthly = []
        for m in MONTHS:
            fname = f"CHELSA_pr_{m}_1981-2010_V.2.1.tif"
            path = os.path.join(CLIMATE_FOLDER_PATHS["pr"], fname)
            if not os.path.exists(path):
                raise FileNotFoundError(f"Missing baseline file: {path}")
            vals = sample_raster(onsseter.df, path)
            baseline_monthly.append(vals)

        baseline_monthly = np.array(baseline_monthly)

        drought_spi = []
        n_points = pr_monthly.shape[1]

        for i in range(n_points):
            spi_val = compute_spi3(baseline_monthly[:, i], pr_monthly[:, i])
            drought_spi.append(spi_val)

        onsseter.df["hazard_drought_spi3"] = drought_spi
        onsseter.df["exposed_drought"] = onsseter.df["hazard_drought_spi3"] <= HAZARD_THRESHOLDS["drought"]
    else:
        onsseter.df["hazard_drought_spi3"] = np.nan
        onsseter.df["exposed_drought"] = False
else:
    onsseter.df["hazard_heat"] = np.nan
    onsseter.df["exposed_heat"] = False
    onsseter.df["hazard_cold"] = np.nan
    onsseter.df["exposed_cold"] = False
    onsseter.df["hazard_high_precip"] = np.nan
    onsseter.df["exposed_high_precip"] = False
    onsseter.df["hazard_drought_spi3"] = np.nan
    onsseter.df["exposed_drought"] = False


# Landslide

if "landslide" in ACTIVE_HAZARDS:
    onsseter.df["hazard_landslide"] = sample_raster(onsseter.df, HAZARD_PATHS["landslide"])

    absolute_floor = 0.01
    percentile_cutoff = onsseter.df["hazard_landslide"].quantile(0.95)
    landslide_threshold = max(absolute_floor, percentile_cutoff)

    onsseter.df["exposed_landslide"] = onsseter.df["hazard_landslide"] > landslide_threshold

    print(f"  Landslide threshold used: {landslide_threshold:.5f}")
else:
    onsseter.df["hazard_landslide"] = np.nan
    onsseter.df["exposed_landslide"] = False


# Check sampling

print("\nHazard sampling complete.\n")

for hazard in ACTIVE_HAZARDS:
    value_col = HAZARD_VALUE_COLUMNS[hazard]
    exposure_col = HAZARD_EXPOSURE_COLUMNS[hazard]

    print(f"{hazard.upper()}")
    print(f"  severity column: {value_col}")
    print(f"  exposure column: {exposure_col}")
    print(f"  exposed settlements: {int(onsseter.df[exposure_col].sum())}")

    if onsseter.df[value_col].notna().any():
        print(f"  min value: {onsseter.df[value_col].min():.3f}")
        print(f"  max value: {onsseter.df[value_col].max():.3f}")
    else:
        print("  no sampled values found")

    print("")

## 7c. Build technology exclusion flags from hazard exposure

In [ ]:
TECHS_FOR_ANALYSIS = ["Grid", "SA_Diesel", "SA_PV", "MG_Diesel", "MG_PV", "MG_Wind", "MG_Hydro"]

def get_exclusion_mask(df, tech, active_hazards, tech_hazard_risk, hazard_exposure_columns, exclusion_risk_threshold):
    """
    Returns:
        exclusion_mask: boolean Series, True where technology should be excluded
        reasons: list of reason strings for each row
    """
    exclusion_mask = pd.Series(False, index=df.index)
    reasons = pd.Series("", index=df.index, dtype="object")

    for hazard in active_hazards:
        risk_score = tech_hazard_risk.get(tech, {}).get(hazard, 0)

        # Exclude only if risk score meets the exclusion threshold
        if risk_score < exclusion_risk_threshold:
            continue

        exposure_col = hazard_exposure_columns[hazard]
        if exposure_col not in df.columns:
            continue

        mask = df[exposure_col].fillna(False)

        exclusion_mask = exclusion_mask | mask
        reasons.loc[mask] = reasons.loc[mask] + f"{hazard}(risk={risk_score});"

    return exclusion_mask, reasons


for tech in TECHS_FOR_ANALYSIS:
    exclude_col = f"exclude_{tech}"
    reason_col = f"exclude_reason_{tech}"

    mask, reasons = get_exclusion_mask(
        onsseter.df,
        tech,
        ACTIVE_HAZARDS,
        TECH_HAZARD_RISK,
        HAZARD_EXPOSURE_COLUMNS,
        EXCLUSION_RISK_THRESHOLD
    )

    onsseter.df[exclude_col] = mask
    onsseter.df[reason_col] = reasons

exclude_cols = [f"exclude_{tech}" for tech in TECHS_FOR_ANALYSIS]
onsseter.df["has_any_exclusion"] = onsseter.df[exclude_cols].any(axis=1)

print("\nTechnology exclusion flags created.\n")
print("Exclusion risk threshold:", EXCLUSION_RISK_THRESHOLD)

for tech in TECHS_FOR_ANALYSIS:
    exclude_col = f"exclude_{tech}"
    reason_col = f"exclude_reason_{tech}"

    excluded_count = int(onsseter.df[exclude_col].sum())
    print(f"{tech}: excluded in {excluded_count} settlements")

    sample_reasons = onsseter.df.loc[onsseter.df[exclude_col], reason_col].head(3).tolist()
    if sample_reasons:
        print("  sample reasons:", sample_reasons)

print("")
print("Settlements with at least one excluded technology:", int(onsseter.df["has_any_exclusion"].sum()))

## 7d. Apply exclusion flags to technology cost columns

In [ ]:
TECH_COST_COLUMN_PREFIX = {
    "Grid": "Grid",
    "SA_Diesel": "SA_Diesel",
    "SA_PV": "SA_PV",
    "MG_Diesel": "MG_Diesel",
    "MG_PV": "MG_PV",
    "MG_Wind": "MG_Wind",
    "MG_Hydro": "MG_Hydro"
}

def apply_exclusions_to_costs(df, year, techs_for_analysis, tech_cost_column_prefix):
   
    # Set cost/LCOE columns to np.inf where a technology is excluded. Returns summary dict with exclusion counts by technology.

    exclusion_summary = {}

    for tech in techs_for_analysis:
        exclude_col = f"exclude_{tech}"
        cost_col = f"{tech_cost_column_prefix[tech]}{year}"

        if exclude_col not in df.columns:
            print(f"[WARN] Missing exclusion column: {exclude_col}")
            continue

        if cost_col not in df.columns:
            print(f"[WARN] Missing technology cost column: {cost_col}")
            continue

        mask = df[exclude_col].fillna(False)
        excluded_count = int(mask.sum())

        df.loc[mask, cost_col] = np.inf
        exclusion_summary[tech] = excluded_count

    return exclusion_summary

## 7e. Climate cost-adjustment settings

In [ ]:
COST_ADJUSTMENT_SCENARIO = "climate_cost"

# Simple percentage adders applied to the technology cost column when a settlement is exposed to a hazard.
# E.g.: 0.10 = +10% cost

TECH_HAZARD_COST_MULTIPLIER = {
    "Grid": {
        "flood": 0.25,
        "fire": 0.15,
        "heat": 0.08,
        "high_precip": 0.10,
        "landslide": 0.20,
        "wind": 0.10
    },
    "SA_Diesel": {
        "flood": 0.10,
        "fire": 0.10,
        "high_precip": 0.05
    },
    "SA_PV": {
        "flood": 0.20,
        "heat": 0.22,
        "high_precip": 0.16,
        "cold": 0.06,
        "fire": 0.10
    },
    "MG_Diesel": {
        "flood": 0.10,
        "fire": 0.10,
        "high_precip": 0.05
    },
    "MG_PV": {
        "flood": 0.12,
        "heat": 0.12,
        "high_precip": 0.08,
        "cold": 0.04,
        "fire": 0.06,
        "landslide": 0.08,
        "wind": 0.05
    },
    "MG_Wind": {
        "wind": 0.06,
        "high_precip": 0.03,
        "fire": 0.04,
        "landslide": 0.06
    },
    "MG_Hydro": {
        "drought": 0.04,
        "flood": 0.02,
        "high_precip": 0.01,
        "landslide": 0.03
    }
}

# Only apply a cost penalty where the technology-hazard interaction score is at least this level
# 1 means any non-zero interaction can receive a penalty.
COST_ADJUSTMENT_RISK_THRESHOLD = 1

print("Climate cost-adjustment settings loaded.")
print("Scenario label:", COST_ADJUSTMENT_SCENARIO)
print("Cost-adjustment risk threshold:", COST_ADJUSTMENT_RISK_THRESHOLD)

## 7f. Apply climate cost adjustments to technology cost columns

In [ ]:
# Combine hazard-specific cost penalties using: max(p_i) + alpha * sum(other p_i)

def combine_cost_penalties_max_plus_remainder(penalties, alpha=0.5, cap=None):

    valid_penalties = [float(p) for p in penalties if pd.notnull(p) and float(p) > 0]

    if not valid_penalties:
        return 0.0

    valid_penalties = sorted(valid_penalties, reverse=True)
    largest = valid_penalties[0]
    others = valid_penalties[1:]

    combined = largest + alpha * sum(others)

    if cap is not None:
        combined = min(combined, cap)

    return combined


def apply_cost_adjustments_to_costs(
    df,
    year,
    techs_for_analysis,
    tech_cost_column_prefix,
    active_hazards,
    tech_hazard_risk,
    tech_hazard_cost_multiplier,
    hazard_exposure_columns,
    cost_adjustment_risk_threshold=1,
    alpha=0.5,
    max_total_penalty_by_tech=None
):

    # Apply percentage cost adders to each technology cost column based on:
    # - settlement hazard exposure
    # - technology-hazard interaction score
    # - technology-hazard cost multiplier
    # Multi-hazard penalties are combined using: max penalty + alpha * sum(other penalties)

    adjustment_summary = {}
    max_total_penalty_by_tech = max_total_penalty_by_tech or {}

    for tech in techs_for_analysis:
        cost_col = f"{tech_cost_column_prefix[tech]}{year}"

        if cost_col not in df.columns:
            print(f"[WARN] Missing technology cost column: {cost_col}")
            continue

        # Make sure column can hold float values
        df[cost_col] = df[cost_col].astype(float)

        total_penalty = pd.Series(0.0, index=df.index)
        reason_text = pd.Series("", index=df.index, dtype="object")

        for idx in df.index:
            hazard_penalties = []
            hazard_reasons = []

            for hazard in active_hazards:
                risk_score = tech_hazard_risk.get(tech, {}).get(hazard, 0)
                penalty = tech_hazard_cost_multiplier.get(tech, {}).get(hazard, 0)

                if risk_score < cost_adjustment_risk_threshold:
                    continue
                if penalty <= 0:
                    continue

                exposure_col = hazard_exposure_columns.get(hazard)
                if exposure_col not in df.columns:
                    continue

                is_exposed = bool(df.at[idx, exposure_col]) if pd.notnull(df.at[idx, exposure_col]) else False
                if not is_exposed:
                    continue

                hazard_penalties.append(penalty)
                hazard_reasons.append(f"{hazard}(+{penalty:.2f})")

            combined_penalty = combine_cost_penalties_max_plus_remainder(
                hazard_penalties,
                alpha=alpha,
                cap=max_total_penalty_by_tech.get(tech, None)
            )

            total_penalty.at[idx] = combined_penalty
            reason_text.at[idx] = ";".join(hazard_reasons) if hazard_reasons else ""

        # Apply multiplier only where penalty > 0
        adjusted_mask = total_penalty > 0
        df.loc[adjusted_mask, cost_col] = (
            df.loc[adjusted_mask, cost_col] * (1 + total_penalty.loc[adjusted_mask])
        )

        # Save diagnostics to dataframe
        df[f"cost_penalty_{tech}_{year}"] = total_penalty
        df[f"cost_penalty_reason_{tech}_{year}"] = reason_text

        adjustment_summary[tech] = {
            "adjusted_settlements": int(adjusted_mask.sum()),
            "mean_penalty": float(total_penalty.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "max_penalty": float(total_penalty.loc[adjusted_mask].max()) if adjusted_mask.any() else 0.0
        }

    return adjustment_summary

## 7g. Climate cost-adjustment v2 settings (CAPEX/OPEX)

In [ ]:
CAPEX_OPEX_ADJUSTMENT_SCENARIO = "climate_cost_v2"

# CAPEX adder represent additional upfront resilience/hardening costs.
# OPEX adder represent additional recurring operation/maintenance burden.

TECH_HAZARD_CAPEX_ADDER = {
    "Grid": {
        "flood": 0.12,
        "fire": 0.08,
        "landslide": 0.10,
        "wind": 0.05
    },
    "SA_Diesel": {
        "flood": 0.04,
        "fire": 0.03
    },
    "SA_PV": {
        "flood": 0.10,
        "fire": 0.05
    },
    "MG_Diesel": {
        "flood": 0.04,
        "fire": 0.03
    },
    "MG_PV": {
        "flood": 0.04,
        "fire": 0.02,
        "landslide": 0.03,
        "wind": 0.02
    },
    "MG_Wind": {
        "wind": 0.05,
        "landslide": 0.03
    },
    "MG_Hydro": {
        "flood": 0.02,
        "landslide": 0.01
    }
}

TECH_HAZARD_OPEX_ADDER = {
    "Grid": {
        "heat": 0.05,
        "high_precip": 0.06,
        "flood": 0.04,
        "fire": 0.04
    },
    "SA_Diesel": {
        "high_precip": 0.03,
        "fire": 0.03
    },
    "SA_PV": {
        "heat": 0.18,
        "high_precip": 0.12,
        "cold": 0.04
    },
    "MG_Diesel": {
        "high_precip": 0.03,
        "fire": 0.03
    },
    "MG_PV": {
        "heat": 0.08,
        "high_precip": 0.05,
        "cold": 0.02,
        "wind": 0.02
    },
    "MG_Wind": {
        "wind": 0.04,
        "high_precip": 0.02
    },
    "MG_Hydro": {
        "drought": 0.03,
        "high_precip": 0.01,
        "flood": 0.01
    }
}

# Only apply v2 adders where the technology-hazard interaction score meets or exceeds this threshold.
CAPEX_OPEX_RISK_THRESHOLD = 1

print("Climate cost-adjustment v2 settings loaded.")
print("Scenario label:", CAPEX_OPEX_ADJUSTMENT_SCENARIO)
print("CAPEX/OPEX risk threshold:", CAPEX_OPEX_RISK_THRESHOLD)

## 7h. Apply climate cost-adjustment v2 (CAPEX/OPEX) to technology cost columns

In [ ]:
def combine_cost_penalties_max_plus_remainder(penalties, alpha=0.5, cap=None):
   
    valid_penalties = [float(p) for p in penalties if pd.notnull(p) and float(p) > 0]

    if not valid_penalties:
        return 0.0

    valid_penalties = sorted(valid_penalties, reverse=True)
    largest = valid_penalties[0]
    others = valid_penalties[1:]

    combined = largest + alpha * sum(others)

    if cap is not None:
        combined = min(combined, cap)

    return combined


def apply_capex_opex_adjustments_to_costs(
    df,
    year,
    techs_for_analysis,
    tech_cost_column_prefix,
    active_hazards,
    tech_hazard_risk,
    tech_hazard_capex_adder,
    tech_hazard_opex_adder,
    hazard_exposure_columns,
    capex_opex_risk_threshold=1,
    opex_discount_factor=1.0,
    alpha=0.5,
    max_capex_penalty_by_tech=None,
    max_opex_penalty_by_tech=None,
    max_total_penalty_by_tech=None
):
   

    adjustment_summary = {}
    max_capex_penalty_by_tech = max_capex_penalty_by_tech or {}
    max_opex_penalty_by_tech = max_opex_penalty_by_tech or {}
    max_total_penalty_by_tech = max_total_penalty_by_tech or {}

    for tech in techs_for_analysis:
        cost_col = f"{tech_cost_column_prefix[tech]}{year}"

        if cost_col not in df.columns:
            print(f"[WARN] Missing technology cost column: {cost_col}")
            continue

        df[cost_col] = df[cost_col].astype(float)

        capex_penalty = pd.Series(0.0, index=df.index)
        opex_penalty = pd.Series(0.0, index=df.index)
        total_penalty = pd.Series(0.0, index=df.index)
        reason_text = pd.Series("", index=df.index, dtype="object")

        for idx in df.index:
            capex_penalties_here = []
            opex_penalties_here = []
            hazard_reasons = []

            for hazard in active_hazards:
                risk_score = tech_hazard_risk.get(tech, {}).get(hazard, 0)

                if risk_score < capex_opex_risk_threshold:
                    continue

                exposure_col = hazard_exposure_columns.get(hazard)
                if exposure_col not in df.columns:
                    continue

                is_exposed = bool(df.at[idx, exposure_col]) if pd.notnull(df.at[idx, exposure_col]) else False
                if not is_exposed:
                    continue

                capex_add = tech_hazard_capex_adder.get(tech, {}).get(hazard, 0.0)
                opex_add = tech_hazard_opex_adder.get(tech, {}).get(hazard, 0.0)

                if capex_add > 0:
                    capex_penalties_here.append(capex_add)
                if opex_add > 0:
                    opex_penalties_here.append(opex_add)

                if capex_add > 0 or opex_add > 0:
                    hazard_reasons.append(f"{hazard}[capex:+{capex_add:.2f},opex:+{opex_add:.2f}]")

            combined_capex = combine_cost_penalties_max_plus_remainder(
                capex_penalties_here,
                alpha=alpha,
                cap=max_capex_penalty_by_tech.get(tech, None)
            )

            combined_opex = combine_cost_penalties_max_plus_remainder(
                opex_penalties_here,
                alpha=alpha,
                cap=max_opex_penalty_by_tech.get(tech, None)
            )

            combined_total = combined_capex + (opex_discount_factor * combined_opex)

            if tech in max_total_penalty_by_tech:
                combined_total = min(combined_total, max_total_penalty_by_tech[tech])

            capex_penalty.at[idx] = combined_capex
            opex_penalty.at[idx] = combined_opex
            total_penalty.at[idx] = combined_total
            reason_text.at[idx] = ";".join(hazard_reasons) if hazard_reasons else ""

        adjusted_mask = total_penalty > 0

        df.loc[adjusted_mask, cost_col] = (
            df.loc[adjusted_mask, cost_col] * (1 + total_penalty.loc[adjusted_mask])
        )

        df[f"capex_penalty_{tech}_{year}"] = capex_penalty
        df[f"opex_penalty_{tech}_{year}"] = opex_penalty
        df[f"total_penalty_v2_{tech}_{year}"] = total_penalty
        df[f"penalty_reason_v2_{tech}_{year}"] = reason_text

        adjustment_summary[tech] = {
            "adjusted_settlements": int(adjusted_mask.sum()),
            "mean_capex_penalty": float(capex_penalty.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "mean_opex_penalty": float(opex_penalty.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "mean_total_penalty": float(total_penalty.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "max_total_penalty": float(total_penalty.loc[adjusted_mask].max()) if adjusted_mask.any() else 0.0
        }

    return adjustment_summary

## 7i. Climate cost-adjustment v3 settings (discounted lifecycle adaptation)

In [ ]:
LIFECYCLE_ADJUSTMENT_SCENARIO = "climate_cost_v3"

# uses the existing discount_rate from the main inputs
LIFECYCLE_DISCOUNT_RATE = discount_rate

# Tech lifetimes for the discounted adaptation calculation (as defined above)

TECH_LIFETIME_V3 = {
    "Grid": 30,
    "SA_Diesel": 10,
    "SA_PV": 15,
    "MG_Diesel": 15,
    "MG_PV": 20,
    "MG_Wind": 20,
    "MG_Hydro": 30
}

# Upfront adaptation cost share (capex) applied once at the start
TECH_HAZARD_UPFRONT_ADAPTATION_SHARE = {
    "Grid": {
        "flood": 0.25,
        "fire": 0.16,
        "landslide": 0.20,
        "wind": 0.12
    },
    "SA_Diesel": {
        "flood": 0.06,
        "fire": 0.05
    },
    "SA_PV": {
        "flood": 0.06,
        "fire": 0.04
    },
    "MG_Diesel": {
        "flood": 0.06,
        "fire": 0.05
    },
    "MG_PV": {
        "flood": 0.08,
        "fire": 0.04,
        "landslide": 0.05,
        "wind": 0.04
    },
    "MG_Wind": {
        "wind": 0.12,
        "landslide": 0.06
    },
    "MG_Hydro": {
        "flood": 0.04,
        "landslide": 0.04
    }
}

# Annual adaptation burden share (opex) discounted over the technology life
TECH_HAZARD_ANNUAL_ADAPTATION_SHARE = {
    "Grid": {
        "heat": 0.006,
        "high_precip": 0.006,
        "fire": 0.004
    },
    "SA_Diesel": {
        "high_precip": 0.004,
        "fire": 0.004
    },
    "SA_PV": {
        "heat": 0.010,
        "high_precip": 0.006,
        "cold": 0.003
    },
    "MG_Diesel": {
        "high_precip": 0.004,
        "fire": 0.004
    },
    "MG_PV": {
        "heat": 0.012,
        "high_precip": 0.008,
        "cold": 0.003,
        "wind": 0.003
    },
    "MG_Wind": {
        "wind": 0.010,
        "high_precip": 0.004
    },
    "MG_Hydro": {
        "drought": 0.005,
        "high_precip": 0.002,
        "flood": 0.001
    }
}

LIFECYCLE_RISK_THRESHOLD = 1

print("Climate cost-adjustment v3 settings loaded.")
print("Scenario label:", LIFECYCLE_ADJUSTMENT_SCENARIO)
print("Using discount rate:", LIFECYCLE_DISCOUNT_RATE)

## 7j. Climate cost-adjustment v3 helper (discounted lifecycle adaptation)

In [ ]:
def discounted_annuity_factor(discount_rate, lifetime_years):
    
    if lifetime_years <= 0:
        return 0.0
    return sum(1 / ((1 + discount_rate) ** t) for t in range(1, lifetime_years + 1))


def combine_cost_penalties_max_plus_remainder(penalties, alpha=0.5, cap=None):

    valid_penalties = [float(p) for p in penalties if pd.notnull(p) and float(p) > 0]

    if not valid_penalties:
        return 0.0

    valid_penalties = sorted(valid_penalties, reverse=True)
    largest = valid_penalties[0]
    others = valid_penalties[1:]

    combined = largest + alpha * sum(others)

    if cap is not None:
        combined = min(combined, cap)

    return combined


def apply_lifecycle_adaptation_adjustments_to_costs(
    df,
    year,
    techs_for_analysis,
    tech_cost_column_prefix,
    active_hazards,
    tech_hazard_risk,
    tech_hazard_upfront_adaptation_share,
    tech_hazard_annual_adaptation_share,
    tech_lifetime_v3,
    hazard_exposure_columns,
    lifecycle_risk_threshold=1,
    discount_rate=0.08,
    alpha=0.5,
    max_upfront_share_by_tech=None,
    max_annual_share_by_tech=None,
    max_total_burden_share_by_tech=None
):

    # Applies discounted lifecycle adaptation burdens to each technology cost column.

    adjustment_summary = {}
    max_upfront_share_by_tech = max_upfront_share_by_tech or {}
    max_annual_share_by_tech = max_annual_share_by_tech or {}
    max_total_burden_share_by_tech = max_total_burden_share_by_tech or {}

    for tech in techs_for_analysis:
        cost_col = f"{tech_cost_column_prefix[tech]}{year}"

        if cost_col not in df.columns:
            print(f"[WARN] Missing technology cost column: {cost_col}")
            continue

        df[cost_col] = df[cost_col].astype(float)

        upfront_share = pd.Series(0.0, index=df.index)
        annual_share = pd.Series(0.0, index=df.index)
        reason_text = pd.Series("", index=df.index, dtype="object")

        lifetime_years = tech_lifetime_v3.get(tech, 20)
        annuity_factor = discounted_annuity_factor(discount_rate, lifetime_years)

        for idx in df.index:
            upfront_penalties_here = []
            annual_penalties_here = []
            hazard_reasons = []

            for hazard in active_hazards:
                risk_score = tech_hazard_risk.get(tech, {}).get(hazard, 0)

                if risk_score < lifecycle_risk_threshold:
                    continue

                exposure_col = hazard_exposure_columns.get(hazard)
                if exposure_col not in df.columns:
                    continue

                is_exposed = bool(df.at[idx, exposure_col]) if pd.notnull(df.at[idx, exposure_col]) else False
                if not is_exposed:
                    continue

                upfront_add = tech_hazard_upfront_adaptation_share.get(tech, {}).get(hazard, 0.0)
                annual_add = tech_hazard_annual_adaptation_share.get(tech, {}).get(hazard, 0.0)

                if upfront_add > 0:
                    upfront_penalties_here.append(upfront_add)
                if annual_add > 0:
                    annual_penalties_here.append(annual_add)

                if upfront_add > 0 or annual_add > 0:
                    hazard_reasons.append(f"{hazard}[upfront:+{upfront_add:.3f},annual:+{annual_add:.3f}]")

            combined_upfront_share = combine_cost_penalties_max_plus_remainder(
                upfront_penalties_here,
                alpha=alpha,
                cap=max_upfront_share_by_tech.get(tech, None)
            )

            combined_annual_share = combine_cost_penalties_max_plus_remainder(
                annual_penalties_here,
                alpha=alpha,
                cap=max_annual_share_by_tech.get(tech, None)
            )

            upfront_share.at[idx] = combined_upfront_share
            annual_share.at[idx] = combined_annual_share
            reason_text.at[idx] = ";".join(hazard_reasons) if hazard_reasons else ""

        base_cost = df[cost_col].copy()
        upfront_burden = base_cost * upfront_share
        discounted_annual_burden = base_cost * annual_share * annuity_factor
        total_burden = upfront_burden + discounted_annual_burden

        # Optional cap on total lifecycle burden as a share of base cost
        if tech in max_total_burden_share_by_tech:
            max_total_burden = base_cost * max_total_burden_share_by_tech[tech]
            total_burden = np.minimum(total_burden, max_total_burden)

        adjusted_mask = total_burden > 0
        df.loc[adjusted_mask, cost_col] = base_cost.loc[adjusted_mask] + total_burden.loc[adjusted_mask]

        df[f"upfront_share_v3_{tech}_{year}"] = upfront_share
        df[f"annual_share_v3_{tech}_{year}"] = annual_share
        df[f"upfront_burden_v3_{tech}_{year}"] = upfront_burden
        df[f"discounted_annual_burden_v3_{tech}_{year}"] = discounted_annual_burden
        df[f"total_burden_v3_{tech}_{year}"] = total_burden
        df[f"penalty_reason_v3_{tech}_{year}"] = reason_text

        adjustment_summary[tech] = {
            "adjusted_settlements": int(adjusted_mask.sum()),
            "mean_upfront_share": float(upfront_share.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "mean_annual_share": float(annual_share.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "mean_total_burden": float(total_burden.loc[adjusted_mask].mean()) if adjusted_mask.any() else 0.0,
            "max_total_burden": float(total_burden.loc[adjusted_mask].max()) if adjusted_mask.any() else 0.0,
            "annuity_factor": float(annuity_factor)
        }

    return adjustment_summary

## 7k. Climate performance-adjustment settings

In [ ]:
PERFORMANCE_ADJUSTMENT_SCENARIO = "climate_performance"

# Multiplicative derating factors applied when a settlement is exposed to a hazard and the technology-hazard interaction is relevant (focus on generation technologies not grid).
# e.g.
# 1.00 = no derating
# 0.95 = 5% reduction in effective performance
# 0.85 = 15% reduction in effective performance

TECH_HAZARD_PERFORMANCE_MULTIPLIER = {
    "SA_PV": {
        "heat": 0.78,
        "high_precip": 0.88,
        "flood": 0.92,
        "cold": 0.95
    },
    "MG_PV": {
        "heat": 0.89,
        "high_precip": 0.93,
        "flood": 0.94,
        "cold": 0.96,
        "wind": 0.96
    },
    "MG_Hydro": {
        "drought": 0.94,
        "high_precip": 0.98,
        "flood": 0.98
    },
    "MG_Wind": {
        "wind": 0.92,
        "high_precip": 0.97,
        "cold": 0.98
    },
    "SA_Diesel": {
        "flood": 0.96,
        "heat": 0.97,
        "fire": 0.96
    },
    "MG_Diesel": {
        "flood": 0.95,
        "heat": 0.97,
        "high_precip": 0.97,
        "fire": 0.96
    }
}

# Only apply a derating where the technology-hazard interaction score meets or exceeds this threshold.
PERFORMANCE_RISK_THRESHOLD = 1

print("Climate performance-adjustment settings loaded.")
print("Scenario label:", PERFORMANCE_ADJUSTMENT_SCENARIO)
print("Performance risk threshold:", PERFORMANCE_RISK_THRESHOLD)

## 7l. Climate performance-adjustment helper

In [ ]:
def apply_performance_adjustments_to_costs(
    df,
    year,
    techs_for_analysis,
    tech_cost_column_prefix,
    active_hazards,
    tech_hazard_risk,
    tech_hazard_performance_multiplier,
    hazard_exposure_columns,
    performance_risk_threshold=1
):

    # Applies performance derating to generator technologies by converting reduced performance into an effective cost increase.

    # Multi-hazard performance effects combined multiplicatively.
    # If combined performance multiplier = m, then: adjusted_cost = base_cost / m

    adjustment_summary = {}

    for tech in techs_for_analysis:
        cost_col = f"{tech_cost_column_prefix[tech]}{year}"

        if cost_col not in df.columns:
            print(f"[WARN] Missing technology cost column: {cost_col}")
            continue

        df[cost_col] = df[cost_col].astype(float)

        # No derating:
        combined_multiplier = pd.Series(1.0, index=df.index)
        reason_text = pd.Series("", index=df.index, dtype="object")

        for hazard in active_hazards:
            risk_score = tech_hazard_risk.get(tech, {}).get(hazard, 0)

            if risk_score < performance_risk_threshold:
                continue

            exposure_col = hazard_exposure_columns.get(hazard)
            if exposure_col not in df.columns:
                continue

            perf_multiplier = tech_hazard_performance_multiplier.get(tech, {}).get(hazard, 1.0)

            # Skip if no derating defined
            if perf_multiplier >= 1.0:
                continue

            mask = df[exposure_col].fillna(False)
            if not mask.any():
                continue

            # Multiplicative stacking of performance effects
            combined_multiplier.loc[mask] *= perf_multiplier
            reason_text.loc[mask] = reason_text.loc[mask] + f"{hazard}[perf:{perf_multiplier:.2f}];"

        adjusted_mask = combined_multiplier < 1.0

        # Lower performance = higher effective cost per delivered unit
        df.loc[adjusted_mask, cost_col] = (
            df.loc[adjusted_mask, cost_col] / combined_multiplier.loc[adjusted_mask]
        )

        df[f"performance_multiplier_{tech}_{year}"] = combined_multiplier
        df[f"performance_reason_{tech}_{year}"] = reason_text

        adjustment_summary[tech] = {
            "adjusted_settlements": int(adjusted_mask.sum()),
            "mean_multiplier": float(combined_multiplier.loc[adjusted_mask].mean()) if adjusted_mask.any() else 1.0,
            "min_multiplier": float(combined_multiplier.loc[adjusted_mask].min()) if adjusted_mask.any() else 1.0
        }

    return adjustment_summary

# 7m. Start a scenario run, which calculate and compare technology costs for every settlement in the country - with baseline, climate-exclusion, and climate-cost cases

Based on the previous calculation this piece of code identifies the LCoE that every off-grid technology can provide, for each single populated settlement of the selected country. The cell then takes all the currently grid-connected points in the country, and looks at the points within a certain distance from them, to see if it is more economical to connect them to the grid, or to use one of the off-grid technologies calculated above. Once more points are connected to the grid, the process is repeated, so that new points close to those points might also be connected. This is repeated until there are no new points to connect to the grid.

In [ ]:
scenario_configs = [
    (BASELINE_SCENARIO, "baseline"),
    (CLIMATE_SCENARIO, "exclude"),
    (COST_ADJUSTMENT_SCENARIO, "adjust_cost"),
    (CAPEX_OPEX_ADJUSTMENT_SCENARIO, "adjust_cost_v2"),
    (LIFECYCLE_ADJUSTMENT_SCENARIO, "adjust_cost_v3"),
    (PERFORMANCE_ADJUSTMENT_SCENARIO, "adjust_performance")
]

techs = ["Grid", "SA_Diesel", "SA_PV", "MG_Diesel", "MG_PV", "MG_Wind", "MG_Hydro"]
tech_codes = [1, 2, 3, 4, 5, 6, 7]

for scenario_label, climate_mode in scenario_configs:
    print("\n==============================")
    print(f"Running scenario: {scenario_label}")
    print("==============================")

    onsseter.current_mv_line_dist()

    try:
        onsseter.df.reset_index(inplace=True)
    except ValueError:
        pass

    for year in yearsofanalysis:
        print(f"\n--- Year {year} | Scenario: {scenario_label} ---")

        prioritization = 2

        eleclimit = eleclimits[year]
        time_step = time_steps[year]
        grid_cap_gen_limit = time_step * annual_grid_cap_gen_limit[year] * 1000
        grid_connect_limit = time_step * annual_new_grid_connections_limit[year] * 1000

        onsseter.set_scenario_variables(
            year,
            num_people_per_hh_rural,
            num_people_per_hh_urban,
            time_step,
            start_year,
            urban_target_tier,
            rural_target_tier,
            end_year_pop,
            productive_demand
        )

        (
            sa_diesel_investment, sa_diesel_capacity,
            sa_pv_investment, sa_pv_capacity,
            mg_diesel_investment, mg_diesel_capacity,
            mg_pv_investment, mg_pv_capacity,
            mg_wind_investment, mg_wind_capacity,
            mg_hydro_investment, mg_hydro_capacity
        ) = onsseter.calculate_off_grid_lcoes(
            mg_hydro_calc,
            mg_wind_calc,
            mg_pv_calc,
            sa_pv_calc,
            mg_diesel_calc,
            sa_diesel_calc,
            year,
            end_year,
            time_step,
            techs,
            tech_codes
        )

        (
            grid_investment,
            grid_capacity,
            grid_cap_gen_limit,
            grid_connect_limit
        ) = onsseter.pre_electrification(
            grid_generation_cost,
            year,
            time_step,
            end_year,
            grid_calc,
            grid_cap_gen_limit,
            grid_connect_limit
        )

        (
            onsseter.df[SET_LCOE_GRID + "{}".format(year)],
            onsseter.df[SET_MIN_GRID_DIST + "{}".format(year)],
            onsseter.df[SET_ELEC_ORDER + "{}".format(year)],
            onsseter.df[SET_MV_CONNECT_DIST],
            grid_investment,
            grid_capacity
        ) = onsseter.elec_extension(
            grid_calc,
            mv_line_max_length,
            year,
            start_year,
            end_year,
            time_step,
            grid_cap_gen_limit,
            grid_connect_limit,
            auto_intensification=auto_intensification,
            prioritization=prioritization,
            new_investment=grid_investment,
            new_capacity=grid_capacity
        )


        # Apply climate method for this scenario

        if climate_mode == "exclude":
            exclusion_summary = apply_exclusions_to_costs(
                onsseter.df,
                year,
                TECHS_FOR_ANALYSIS,
                TECH_COST_COLUMN_PREFIX
            )
        
            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = onsseter.df["has_any_exclusion"].copy()
        
            print(f"\n[INFO] Applied exclusions for {scenario_label} {year}")
            for tech, count in exclusion_summary.items():
                print(f"  {tech}: {count} excluded settlements")
        
        elif climate_mode == "adjust_cost":
            cost_adjustment_summary = apply_cost_adjustments_to_costs(
                df=onsseter.df,
                year=year,
                techs_for_analysis=TECHS_FOR_ANALYSIS,
                tech_cost_column_prefix=TECH_COST_COLUMN_PREFIX,
                active_hazards=ACTIVE_HAZARDS,
                tech_hazard_risk=TECH_HAZARD_RISK,
                tech_hazard_cost_multiplier=TECH_HAZARD_COST_MULTIPLIER,
                hazard_exposure_columns=HAZARD_EXPOSURE_COLUMNS,
                cost_adjustment_risk_threshold=COST_ADJUSTMENT_RISK_THRESHOLD,
                alpha=0.5
            )
        
            print(f"\n[INFO] Applied cost adjustments for {scenario_label} {year}")
            for tech, stats in cost_adjustment_summary.items():
                print(
                    f"  {tech}: adjusted_settlements={stats['adjusted_settlements']}, "
                    f"mean_penalty={stats['mean_penalty']:.3f}, "
                    f"max_penalty={stats['max_penalty']:.3f}"
                )
        
            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = False

        elif climate_mode == "adjust_cost_v2":
            cost_v2_summary = apply_capex_opex_adjustments_to_costs(
                df=onsseter.df,
                year=year,
                techs_for_analysis=TECHS_FOR_ANALYSIS,
                tech_cost_column_prefix=TECH_COST_COLUMN_PREFIX,
                active_hazards=ACTIVE_HAZARDS,
                tech_hazard_risk=TECH_HAZARD_RISK,
                tech_hazard_capex_adder=TECH_HAZARD_CAPEX_ADDER,
                tech_hazard_opex_adder=TECH_HAZARD_OPEX_ADDER,
                hazard_exposure_columns=HAZARD_EXPOSURE_COLUMNS,
                capex_opex_risk_threshold=CAPEX_OPEX_RISK_THRESHOLD,
                opex_discount_factor=1.0,
                alpha=0.5
            )

            print(f"\n[INFO] Applied CAPEX/OPEX cost adjustments for {scenario_label} {year}")
            for tech, stats in cost_v2_summary.items():
                print(
                    f"  {tech}: adjusted_settlements={stats['adjusted_settlements']}, "
                    f"mean_capex_penalty={stats['mean_capex_penalty']:.3f}, "
                    f"mean_opex_penalty={stats['mean_opex_penalty']:.3f}, "
                    f"mean_total_penalty={stats['mean_total_penalty']:.3f}, "
                    f"max_total_penalty={stats['max_total_penalty']:.3f}"
                )
        
            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = False

        elif climate_mode == "adjust_cost_v3":
            cost_v3_summary = apply_lifecycle_adaptation_adjustments_to_costs(
                df=onsseter.df,
                year=year,
                techs_for_analysis=TECHS_FOR_ANALYSIS,
                tech_cost_column_prefix=TECH_COST_COLUMN_PREFIX,
                active_hazards=ACTIVE_HAZARDS,
                tech_hazard_risk=TECH_HAZARD_RISK,
                tech_hazard_upfront_adaptation_share=TECH_HAZARD_UPFRONT_ADAPTATION_SHARE,
                tech_hazard_annual_adaptation_share=TECH_HAZARD_ANNUAL_ADAPTATION_SHARE,
                tech_lifetime_v3=TECH_LIFETIME_V3,
                hazard_exposure_columns=HAZARD_EXPOSURE_COLUMNS,
                lifecycle_risk_threshold=LIFECYCLE_RISK_THRESHOLD,
                discount_rate=LIFECYCLE_DISCOUNT_RATE,
                alpha=0.5
            )
        
            print(f"\n[INFO] Applied lifecycle adaptation adjustments for {scenario_label} {year}")
            for tech, stats in cost_v3_summary.items():
                print(
                    f"  {tech}: adjusted_settlements={stats['adjusted_settlements']}, "
                    f"mean_upfront_share={stats['mean_upfront_share']:.3f}, "
                    f"mean_annual_share={stats['mean_annual_share']:.3f}, "
                    f"mean_total_burden={stats['mean_total_burden']:.3f}, "
                    f"max_total_burden={stats['max_total_burden']:.3f}, "
                    f"annuity_factor={stats['annuity_factor']:.3f}"
                )

            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = False

        elif climate_mode == "adjust_performance":
            performance_summary = apply_performance_adjustments_to_costs(
                df=onsseter.df,
                year=year,
                techs_for_analysis=TECHS_FOR_ANALYSIS,
                tech_cost_column_prefix=TECH_COST_COLUMN_PREFIX,
                active_hazards=ACTIVE_HAZARDS,
                tech_hazard_risk=TECH_HAZARD_RISK,
                tech_hazard_performance_multiplier=TECH_HAZARD_PERFORMANCE_MULTIPLIER,
                hazard_exposure_columns=HAZARD_EXPOSURE_COLUMNS,
                performance_risk_threshold=PERFORMANCE_RISK_THRESHOLD
            )
        
            print(f"\n[INFO] Applied performance adjustments for {scenario_label} {year}")
            for tech, stats in performance_summary.items():
                print(
                    f"  {tech}: adjusted_settlements={stats['adjusted_settlements']}, "
                    f"mean_multiplier={stats['mean_multiplier']:.3f}, "
                    f"min_multiplier={stats['min_multiplier']:.3f}"
                )
        
            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = False
        
        else:
            onsseter.df[f"has_any_exclusion_{year}_{scenario_label}"] = False

        onsseter.results_columns(
            techs,
            tech_codes,
            year,
            time_step,
            prioritization,
            auto_intensification
        )

        onsseter.calculate_investments_and_capacity(
            sa_diesel_investment, sa_diesel_capacity,
            sa_pv_investment, sa_pv_capacity,
            mg_diesel_investment, mg_diesel_capacity,
            mg_pv_investment, mg_pv_capacity,
            mg_wind_investment, mg_wind_capacity,
            mg_hydro_investment, mg_hydro_capacity,
            grid_investment, grid_capacity, year
        )

        onsseter.apply_limitations(
            eleclimit,
            year,
            time_step,
            prioritization,
            auto_intensification
        )


        # Save scenario-specific outputs

        onsseter.df[f"FinalElecCode{year}_{scenario_label}"] = onsseter.df[f"FinalElecCode{year}"].copy()

        # Save scenario-specific results columns for comparison
        onsseter.df[f'InvestmentCost{year}_{scenario_label}'] = onsseter.df[f'InvestmentCost{year}'].copy()
        onsseter.df[f'NewCapacity{year}_{scenario_label}'] = onsseter.df[f'NewCapacity{year}'].copy()
        onsseter.df[f'NewConnections{year}_{scenario_label}'] = onsseter.df[f'NewConnections{year}'].copy()
        onsseter.df[f'Pop{year}_{scenario_label}'] = onsseter.df[f'Pop{year}'].copy()

        if f"MinimumOverall{year}" in onsseter.df.columns:
            onsseter.df[f"MinimumOverall{year}_{scenario_label}"] = onsseter.df[f"MinimumOverall{year}"].copy()

        print(f"{scenario_label} {year} complete.")

print("\nAll scenario runs complete.")

## 7n. Compare baseline and climate-exclusion results

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
clim_col = f"FinalElecCode{year}_{CLIMATE_SCENARIO}"
excl_col = f"has_any_exclusion_{year}_{CLIMATE_SCENARIO}"

print("Checking comparison columns:")
print(base_col, base_col in onsseter.df.columns)
print(clim_col, clim_col in onsseter.df.columns)
print(excl_col, excl_col in onsseter.df.columns)

if base_col not in onsseter.df.columns or clim_col not in onsseter.df.columns:
    raise KeyError("Baseline or climate-exclusion final code columns are missing. Re-run Section 7e first.")

onsseter.df["tech_changed"] = onsseter.df[base_col] != onsseter.df[clim_col]

if excl_col in onsseter.df.columns:
    onsseter.df["reassigned_due_to_exclusion"] = onsseter.df["tech_changed"] & onsseter.df[excl_col].fillna(False)
else:
    onsseter.df["reassigned_due_to_exclusion"] = onsseter.df["tech_changed"]

print("\nComparison summary:")
print("Changed settlements:", int(onsseter.df["tech_changed"].sum()))
print("Reassigned due to exclusion:", int(onsseter.df["reassigned_due_to_exclusion"].sum()))

print("\nBaseline final tech counts:")
print(onsseter.df[base_col].value_counts(dropna=False).sort_index())

print("\nClimate final tech counts:")
print(onsseter.df[clim_col].value_counts(dropna=False).sort_index())

print("\nChange matrix (baseline -> climate):")
change_matrix = pd.crosstab(
    onsseter.df[base_col],
    onsseter.df[clim_col],
    rownames=["Baseline"],
    colnames=["Climate exclusion"],
    dropna=False
)
print(change_matrix)

## 7o. Compare baseline and climate-cost results

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_col = f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}"

print("Checking comparison columns:")
print(base_col, base_col in onsseter.df.columns)
print(cost_col, cost_col in onsseter.df.columns)

if base_col not in onsseter.df.columns or cost_col not in onsseter.df.columns:
    raise KeyError("Baseline or climate-cost final code columns are missing. Re-run Section 7e first.")

onsseter.df["tech_changed_cost"] = onsseter.df[base_col] != onsseter.df[cost_col]

print("\nComparison summary:")
print("Changed settlements:", int(onsseter.df["tech_changed_cost"].sum()))

print("\nBaseline final tech counts:")
print(onsseter.df[base_col].value_counts(dropna=False).sort_index())

print("\nClimate-cost final tech counts:")
print(onsseter.df[cost_col].value_counts(dropna=False).sort_index())

print("\nChange matrix (baseline -> climate cost):")
change_matrix_cost = pd.crosstab(
    onsseter.df[base_col],
    onsseter.df[cost_col],
    rownames=["Baseline"],
    colnames=["Climate cost"],
    dropna=False
)
print(change_matrix_cost)

## 7p. Compare baseline and climate-cost v2 results

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v2_col = f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}"

print("Checking comparison columns:")
print(base_col, base_col in onsseter.df.columns)
print(cost_v2_col, cost_v2_col in onsseter.df.columns)

if base_col not in onsseter.df.columns or cost_v2_col not in onsseter.df.columns:
    raise KeyError("Baseline or climate-cost-v2 final code columns are missing. Re-run the scenario block first.")

onsseter.df["tech_changed_cost_v2"] = onsseter.df[base_col] != onsseter.df[cost_v2_col]

print("\nComparison summary:")
print("Changed settlements:", int(onsseter.df["tech_changed_cost_v2"].sum()))

print("\nBaseline final tech counts:")
print(onsseter.df[base_col].value_counts(dropna=False).sort_index())

print("\nClimate-cost-v2 final tech counts:")
print(onsseter.df[cost_v2_col].value_counts(dropna=False).sort_index())

print("\nChange matrix (baseline -> climate cost v2):")
change_matrix_cost_v2 = pd.crosstab(
    onsseter.df[base_col],
    onsseter.df[cost_v2_col],
    rownames=["Baseline"],
    colnames=["Climate cost v2"],
    dropna=False
)
print(change_matrix_cost_v2)

## 7q. Compare baseline and climate-cost v3 results

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v3_col = f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}"

print("Checking comparison columns:")
print(base_col, base_col in onsseter.df.columns)
print(cost_v3_col, cost_v3_col in onsseter.df.columns)

if base_col not in onsseter.df.columns or cost_v3_col not in onsseter.df.columns:
    raise KeyError("Baseline or climate-cost-v3 final code columns are missing. Re-run the scenario block first.")

onsseter.df["tech_changed_cost_v3"] = onsseter.df[base_col] != onsseter.df[cost_v3_col]

print("\nComparison summary:")
print("Changed settlements:", int(onsseter.df["tech_changed_cost_v3"].sum()))

print("\nBaseline final tech counts:")
print(onsseter.df[base_col].value_counts(dropna=False).sort_index())

print("\nClimate-cost-v3 final tech counts:")
print(onsseter.df[cost_v3_col].value_counts(dropna=False).sort_index())

print("\nChange matrix (baseline -> climate cost v3):")
change_matrix_cost_v3 = pd.crosstab(
    onsseter.df[base_col],
    onsseter.df[cost_v3_col],
    rownames=["Baseline"],
    colnames=["Climate cost v3"],
    dropna=False
)
print(change_matrix_cost_v3)

## 7r. Compare baseline and climate-performance results

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
perf_col = f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"

print("Checking comparison columns:")
print(base_col, base_col in onsseter.df.columns)
print(perf_col, perf_col in onsseter.df.columns)

if base_col not in onsseter.df.columns or perf_col not in onsseter.df.columns:
    raise KeyError("Baseline or climate-performance final code columns are missing. Re-run the scenario block first.")

onsseter.df["tech_changed_performance"] = onsseter.df[base_col] != onsseter.df[perf_col]

print("\nComparison summary:")
print("Changed settlements:", int(onsseter.df["tech_changed_performance"].sum()))

print("\nBaseline final tech counts:")
print(onsseter.df[base_col].value_counts(dropna=False).sort_index())

print("\nClimate-performance final tech counts:")
print(onsseter.df[perf_col].value_counts(dropna=False).sort_index())

print("\nChange matrix (baseline -> climate performance):")
change_matrix_performance = pd.crosstab(
    onsseter.df[base_col],
    onsseter.df[perf_col],
    rownames=["Baseline"],
    colnames=["Climate performance"],
    dropna=False
)
print(change_matrix_performance)

# 8. Results, Summaries and Visualization
With all the calculations and grid-extensions complete, this block gets the final results on which technology was chosen for each point, how much capacity needs to be installed and what it will cost. Then the summaries, plots and maps are generated.

In [ ]:
elements = []
for year in yearsofanalysis:
    elements.append("Population{}".format(year))
    elements.append("NewConnections{}".format(year))
    elements.append("Capacity{}".format(year))
    elements.append("Investment{}".format(year))

sumtechs = []
for year in yearsofanalysis:
    sumtechs.extend(["Population{}".format(year) + t for t in techs])
    sumtechs.extend(["NewConnections{}".format(year) + t for t in techs])
    sumtechs.extend(["Capacity{}".format(year) + t for t in techs])
    sumtechs.extend(["Investment{}".format(year) + t for t in techs])

summary = pd.Series(index=sumtechs, name='country', dtype='float64')

for year in yearsofanalysis:
    for t in techs:
        summary.loc["Population{}".format(year) + t] = onsseter.df.loc[(onsseter.df[SET_MIN_OVERALL + '{}'.format(year)] == t + '{}'.format(year)) & (onsseter.df[SET_ELEC_FINAL_CODE + '{}'.format(year)] < 99), SET_POP + '{}'.format(year)].sum()
        summary.loc["NewConnections{}".format(year) + t] = onsseter.df.loc[(onsseter.df[SET_MIN_OVERALL + '{}'.format(year)] == t + '{}'.format(year)) & (onsseter.df[SET_ELEC_FINAL_CODE + '{}'.format(year)] < 99), SET_NEW_CONNECTIONS + '{}'.format(year)].sum()
        summary.loc["Capacity{}".format(year) + t] = onsseter.df.loc[(onsseter.df[SET_MIN_OVERALL + '{}'.format(year)] == t + '{}'.format(year)) & (onsseter.df[SET_ELEC_FINAL_CODE + '{}'.format(year)] < 99), SET_NEW_CAPACITY + '{}'.format(year)].sum()/1000
        summary.loc["Investment{}".format(year) + t] = onsseter.df.loc[(onsseter.df[SET_MIN_OVERALL + '{}'.format(year)] == t + '{}'.format(year)) & (onsseter.df[SET_ELEC_FINAL_CODE + '{}'.format(year)] < 99), SET_INVESTMENT_COST + '{}'.format(year)].sum()
        
index = techs + ['Total']
columns = []
for year in yearsofanalysis:
    columns.append("Population{}".format(year))
    columns.append("NewConnections{}".format(year))
    columns.append("Capacity{} (MW)".format(year))
    columns.append("Investment{} (million USD)".format(year))
                                                                                                                                           
summary_table = pd.DataFrame(index=index, columns=columns)

summary_table[columns[0]] = summary.iloc[0:7].astype(int).tolist() + [int(summary.iloc[0:7].sum())]
summary_table[columns[1]] = summary.iloc[7:14].astype(int).tolist() + [int(summary.iloc[7:14].sum())]
summary_table[columns[2]] = summary.iloc[14:21].astype(int).tolist() + [int(summary.iloc[14:21].sum())]
summary_table[columns[3]] = [round(x/1e4)/1e2 for x in summary.iloc[21:28].astype(float).tolist()] + [round(summary.iloc[21:28].sum()/1e4)/1e2]
summary_table[columns[4]] = summary.iloc[28:35].astype(int).tolist() + [int(summary.iloc[28:35].sum())]
summary_table[columns[5]] = summary.iloc[35:42].astype(int).tolist() + [int(summary.iloc[35:42].sum())]
summary_table[columns[6]] = summary.iloc[42:49].astype(int).tolist() + [int(summary.iloc[42:49].sum())]
summary_table[columns[7]] = [round(x/1e4)/1e2 for x in summary.iloc[49:56].astype(float).tolist()] + [round(summary.iloc[49:56].sum()/1e4)/1e2]

display(Markdown('### National summary for currently loaded scenario'))
summary_table

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

TECH_LABELS = {
    1: "Grid",
    2: "SA_Diesel",
    3: "SA_PV",
    4: "MG_Diesel",
    5: "MG_PV",
    6: "MG_Wind",
    7: "MG_Hydro"
}

TECH_ORDER = ["Grid", "SA_Diesel", "SA_PV", "MG_Diesel", "MG_PV", "MG_Wind", "MG_Hydro"]

SCENARIO_LABELS = {
    "baseline": "Baseline",
    "climate_exclusion": "Climate exclusion",
    "climate_cost": "Climate cost v1",
    "climate_cost_v2": "Climate cost v2",
    "climate_cost_v3": "Climate cost v3",
    "climate_performance": "Climate performance"
}

def format_table_for_display(df, integer_cols=None, float_cols=None, decimals=2):
    out = df.copy()
    integer_cols = integer_cols or []
    float_cols = float_cols or []
    for c in integer_cols:
        if c in out.columns:
            out[c] = out[c].map(lambda x: f"{int(x):,}" if pd.notnull(x) else "")
    for c in float_cols:
        if c in out.columns:
            out[c] = out[c].map(lambda x: f"{x:,.{decimals}f}" if pd.notnull(x) else "")
    return out

In [ ]:
# All scenario comparison summary

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
excl_col = f"FinalElecCode{year}_{CLIMATE_SCENARIO}"
cost_col = f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}"
cost_v2_col = f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}"
cost_v3_col = f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}"
perf_col = f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"

comparison_summary = pd.DataFrame({
    "Baseline": onsseter.df[base_col].value_counts().sort_index(),
    "Climate exclusion": onsseter.df[excl_col].value_counts().sort_index(),
    "Climate cost v1": onsseter.df[cost_col].value_counts().sort_index(),
    "Climate cost v2": onsseter.df[cost_v2_col].value_counts().sort_index(),
    "Climate cost v3": onsseter.df[cost_v3_col].value_counts().sort_index(),
    "Climate performance": onsseter.df[perf_col].value_counts().sort_index()
}).fillna(0).astype(int)

comparison_summary.index = comparison_summary.index.map({
    1.0: "Grid",
    2.0: "SA_Diesel",
    3.0: "SA_PV",
    4.0: "MG_Diesel",
    5.0: "MG_PV",
    6.0: "MG_Wind",
    7.0: "MG_Hydro"
})

print("Changed settlements - exclusion:", int(onsseter.df["tech_changed"].sum()))
print("Changed settlements - cost v1:", int(onsseter.df["tech_changed_cost"].sum()))
print("Changed settlements - cost v2:", int(onsseter.df["tech_changed_cost_v2"].sum()))
print("Changed settlements - cost v3:", int(onsseter.df["tech_changed_cost_v3"].sum()))
print("Changed settlements - performance:", int(onsseter.df["tech_changed_performance"].sum()))
print("\nTechnology counts by scenario:")
display(comparison_summary)

comparison_summary_display = comparison_summary.reindex(TECH_ORDER).fillna(0).astype(int).reset_index()
comparison_summary_display = comparison_summary_display.rename(columns={"index": "Technology"})

display(Markdown("### Technology counts by scenario"))
display(format_table_for_display(
    comparison_summary_display,
    integer_cols=["Baseline", "Climate exclusion", "Climate cost v1", "Climate cost v2", "Climate cost v3", "Climate performance"]
))

In [ ]:
# Population by technology and scenario (2030)

year = end_year

scenario_cols = {
    "Baseline": f"FinalElecCode{year}_{BASELINE_SCENARIO}",
    "Climate exclusion": f"FinalElecCode{year}_{CLIMATE_SCENARIO}",
    "Climate cost v1": f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}",
    "Climate cost v2": f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}",
    "Climate cost v3": f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}",
    "Climate performance": f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"
}

tech_code_to_name = {
    1: "Grid",
    2: "SA_Diesel",
    3: "SA_PV",
    4: "MG_Diesel",
    5: "MG_PV",
    6: "MG_Wind",
    7: "MG_Hydro"
}

tech_order = ["Grid", "SA_Diesel", "SA_PV", "MG_Diesel", "MG_PV", "MG_Wind", "MG_Hydro"]

population_mix_rows = []

for scenario_name_label, final_code_col in scenario_cols.items():
    if final_code_col not in onsseter.df.columns:
        continue

    row = {"Scenario": scenario_name_label}

    for code, tech_name in tech_code_to_name.items():
        mask = onsseter.df[final_code_col] == code
        row[tech_name] = onsseter.df.loc[mask, f"Pop{year}"].sum()

    population_mix_rows.append(row)

population_mix_df = pd.DataFrame(population_mix_rows).set_index("Scenario")
population_mix_df = population_mix_df.reindex(columns=tech_order).fillna(0)

population_mix_display = population_mix_df.reset_index()
display(Markdown(f"### Population by technology and scenario ({year})"))
display(format_table_for_display(
    population_mix_display,
    float_cols=TECH_ORDER,
    decimals=0
))

fig_population_mix, ax_population_mix = plt.subplots(figsize=(12, 7))
population_mix_df.plot(kind="bar", stacked=True, ax=ax_population_mix)
ax_population_mix.set_title(f"Population by technology and scenario ({year})")
ax_population_mix.set_ylabel("Population")
ax_population_mix.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Scenario metrics table

year = end_year

scenario_cols = {
    "Baseline": f"FinalElecCode{year}_{BASELINE_SCENARIO}",
    "Climate exclusion": f"FinalElecCode{year}_{CLIMATE_SCENARIO}",
    "Climate cost v1": f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}",
    "Climate cost v2": f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}",
    "Climate cost v3": f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}",
    "Climate performance": f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"
}

scenario_metrics = []

for scenario_name_label, final_code_col in scenario_cols.items():
    if final_code_col not in onsseter.df.columns:
        continue

    electrified_mask = onsseter.df[final_code_col].isin(TECH_LABELS.keys())

    total_population = onsseter.df.loc[electrified_mask, f"Pop{year}"].sum()
    total_new_connections = onsseter.df.loc[electrified_mask, f"NewConnections{year}"].sum()
    total_capacity_mw = onsseter.df.loc[electrified_mask, f"NewCapacity{year}"].sum() / 1000
    total_investment_musd = onsseter.df.loc[electrified_mask, f"InvestmentCost{year}"].sum() / 1e6

    scenario_metrics.append({
        "Scenario": scenario_name_label,
        "Population": total_population,
        "NewConnections": total_new_connections,
        "Capacity_MW": total_capacity_mw,
        "Investment_MUSD": total_investment_musd
    })

scenario_metrics_df = pd.DataFrame(scenario_metrics)

scenario_metrics_df["Population_M"] = scenario_metrics_df["Population"] / 1e6
scenario_metrics_df["NewConnections_M"] = scenario_metrics_df["NewConnections"] / 1e6
scenario_metrics_df["Investment_per_connection_USD"] = (
    scenario_metrics_df["Investment_MUSD"] * 1e6 / scenario_metrics_df["NewConnections"].replace(0, np.nan)
)
scenario_metrics_df["Investment_per_person_USD"] = (
    scenario_metrics_df["Investment_MUSD"] * 1e6 / scenario_metrics_df["Population"].replace(0, np.nan)
)

scenario_metrics_display = scenario_metrics_df[[
    "Scenario", "Population_M", "NewConnections_M", "Capacity_MW",
    "Investment_MUSD", "Investment_per_connection_USD", "Investment_per_person_USD"
]].copy()

display(Markdown(f"### Scenario metrics ({year})"))
display(format_table_for_display(
    scenario_metrics_display,
    float_cols=[
        "Population_M", "NewConnections_M", "Capacity_MW",
        "Investment_MUSD", "Investment_per_connection_USD", "Investment_per_person_USD"
    ],
    decimals=2
))

In [ ]:
scenario_delta_df = scenario_metrics_df.set_index("Scenario").copy()
baseline_metrics = scenario_delta_df.loc["Baseline"]
delta_metrics_df = scenario_delta_df.subtract(baseline_metrics, axis=1).drop(index="Baseline")

fig_delta_metrics, axes = plt.subplots(2, 2, figsize=(14, 10))

delta_metrics_to_plot = [
    ("Population_M", "Population difference vs baseline (millions)"),
    ("NewConnections_M", "New connections difference vs baseline (millions)"),
    ("Capacity_MW", "Capacity difference vs baseline (MW)"),
    ("Investment_MUSD", "Investment difference vs baseline (million USD)")
]

for ax, (metric_col, title) in zip(axes.flatten(), delta_metrics_to_plot):
    ax.bar(delta_metrics_df.index, delta_metrics_df[metric_col])
    ax.axhline(0, linewidth=1)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
settlement_mix_rows = []

for scenario_name_label, final_code_col in scenario_cols.items():
    row = {"Scenario": scenario_name_label}
    for code, tech_name in TECH_LABELS.items():
        row[tech_name] = int((onsseter.df[final_code_col] == code).sum())
    settlement_mix_rows.append(row)

settlement_mix_df = pd.DataFrame(settlement_mix_rows).set_index("Scenario")
settlement_mix_df = settlement_mix_df.reindex(columns=TECH_ORDER).fillna(0)

settlement_mix_share_df = settlement_mix_df.div(settlement_mix_df.sum(axis=1), axis=0) * 100

fig_settlement_share, ax_settlement_share = plt.subplots(figsize=(12, 7))
settlement_mix_share_df.plot(kind="bar", stacked=True, ax=ax_settlement_share)
ax_settlement_share.set_title(f"Settlement share by technology and scenario ({year})")
ax_settlement_share.set_ylabel("Share of settlements (%)")
ax_settlement_share.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
population_mix_share_df = population_mix_df.div(population_mix_df.sum(axis=1), axis=0) * 100

fig_population_share, ax_population_share = plt.subplots(figsize=(12, 7))
population_mix_share_df.plot(kind="bar", stacked=True, ax=ax_population_share)
ax_population_share.set_title(f"Population share by technology and scenario ({year})")
ax_population_share.set_ylabel("Share of electrified population (%)")
ax_population_share.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Combined figure: changes vs baseline
# Left = settlement counts
# Right = population served

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Settlement-count differences

comparison_summary_delta = comparison_summary.subtract(comparison_summary["Baseline"], axis=0).drop(columns=["Baseline"])
comparison_summary_delta = comparison_summary_delta.reindex(TECH_ORDER)
comparison_summary_delta = comparison_summary_delta.apply(pd.to_numeric, errors="coerce")


# Population differences
# population_mix_df:
#   index = scenarios
#   columns = technologies

population_mix_delta_df = population_mix_df.subtract(population_mix_df.loc["Baseline"], axis=1).drop(index="Baseline")
population_mix_delta_df = population_mix_delta_df.reindex(columns=TECH_ORDER)
population_mix_delta_df = population_mix_delta_df.apply(pd.to_numeric, errors="coerce")

# Scenario order

scenario_order = [
    "Climate exclusion",
    "Climate cost v1",
    "Climate cost v2",
    "Climate cost v3",
    "Climate performance"
]

available_scenario_cols = [c for c in scenario_order if c in comparison_summary_delta.columns]
available_scenario_rows = [r for r in scenario_order if r in population_mix_delta_df.index]

settlement_panel = comparison_summary_delta.loc[:, available_scenario_cols]
population_panel = population_mix_delta_df.loc[available_scenario_rows, TECH_ORDER].T / 1e6

settlement_vals = settlement_panel.to_numpy(dtype=float)
population_vals = population_panel.to_numpy(dtype=float)

settlement_vmax = np.nanmax(np.abs(settlement_vals))
population_vmax = np.nanmax(np.abs(population_vals))

if not np.isfinite(settlement_vmax) or settlement_vmax == 0:
    settlement_vmax = 1
if not np.isfinite(population_vmax) or population_vmax == 0:
    population_vmax = 1

fig_combined_delta, axes = plt.subplots(1, 2, figsize=(16, 6))


# Panel 1: settlement count change

im1 = axes[0].imshow(
    settlement_vals,
    aspect='auto',
    cmap='RdBu',
    vmin=-settlement_vmax,
    vmax=settlement_vmax
)

axes[0].set_title(f"Change in settlement counts vs baseline ({year})")
axes[0].set_xticks(range(len(settlement_panel.columns)))
axes[0].set_xticklabels(settlement_panel.columns, rotation=45, ha='right')
axes[0].set_yticks(range(len(settlement_panel.index)))
axes[0].set_yticklabels(settlement_panel.index)
axes[0].set_xlabel("Scenario")
axes[0].set_ylabel("Technology")

for i in range(settlement_panel.shape[0]):
    for j in range(settlement_panel.shape[1]):
        val = settlement_panel.iloc[i, j]
        label = "" if pd.isna(val) or val == 0 else f"{val:+,.0f}"
        axes[0].text(j, i, label, ha='center', va='center', fontsize=9)

fig_combined_delta.colorbar(
    im1, ax=axes[0], fraction=0.046, pad=0.04, label="Difference in settlements"
)


# Panel 2: population served change

im2 = axes[1].imshow(
    population_vals,
    aspect='auto',
    cmap='RdBu',
    vmin=-population_vmax,
    vmax=population_vmax
)

axes[1].set_title(f"Change in population served vs baseline ({year})")
axes[1].set_xticks(range(len(population_panel.columns)))
axes[1].set_xticklabels(population_panel.columns, rotation=45, ha='right')
axes[1].set_yticks(range(len(population_panel.index)))
axes[1].set_yticklabels(population_panel.index)
axes[1].set_xlabel("Scenario")
axes[1].set_ylabel("Technology")

for i in range(population_panel.shape[0]):
    for j in range(population_panel.shape[1]):
        val = population_panel.iloc[i, j]
        label = "" if pd.isna(val) or abs(val) < 1e-9 else f"{val:+.2f}"
        axes[1].text(j, i, label, ha='center', va='center', fontsize=9)

fig_combined_delta.colorbar(
    im2, ax=axes[1], fraction=0.046, pad=0.04, label="Difference in population (million people)"
)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: test shift between two technologies (specificy in code)

pv_switch_df = comparison_summary.loc[["Grid", "SA_PV"]].T.copy()
pv_switch_delta_df = pv_switch_df.subtract(pv_switch_df.loc["Baseline"], axis=1).drop(index="Baseline")

fig_pv_switch, ax_pv_switch = plt.subplots(figsize=(10, 6))
pv_switch_delta_df.plot(kind="bar", ax=ax_pv_switch)
ax_pv_switch.axhline(0, linewidth=1)
ax_pv_switch.set_title(f"Shift between SA_PV and MG_PV relative to baseline ({year})")
ax_pv_switch.set_ylabel("Change in number of settlements")
ax_pv_switch.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
pop_col = f"Pop{year}"
base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"

shift_summary = []

scenario_shift_cols = {
    "Climate exclusion": f"FinalElecCode{year}_{CLIMATE_SCENARIO}",
    "Climate cost v1": f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}",
    "Climate cost v2": f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}",
    "Climate cost v3": f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}",
    "Climate performance": f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"
}

for scenario_name_label, scen_col in scenario_shift_cols.items():
    changed_mask = onsseter.df[base_col] != onsseter.df[scen_col]
    shifted_population = onsseter.df.loc[changed_mask, pop_col].sum()
    shift_summary.append({
        "Scenario": scenario_name_label,
        "Population shifted from baseline": shifted_population,
        "Population shifted from baseline (millions)": shifted_population / 1e6
    })

shift_summary_df = pd.DataFrame(shift_summary)

display(Markdown(f"### Population shifted from baseline ({year})"))
display(format_table_for_display(
    shift_summary_df,
    float_cols=["Population shifted from baseline (millions)"],
    integer_cols=["Population shifted from baseline"],
    decimals=2
))

fig_shift_population, ax_shift_population = plt.subplots(figsize=(10, 6))
ax_shift_population.bar(
    shift_summary_df["Scenario"],
    shift_summary_df["Population shifted from baseline (millions)"]
)
ax_shift_population.set_title(f"Population shifted from baseline by scenario ({year})")
ax_shift_population.set_ylabel("Population shifted (millions of people)")
ax_shift_population.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig_inv_per_conn, ax_inv_per_conn = plt.subplots(figsize=(9, 5))
ax_inv_per_conn.bar(scenario_metrics_df["Scenario"], scenario_metrics_df["Investment_per_connection_USD"])
ax_inv_per_conn.set_title(f"Investment per new connection ({year})")
ax_inv_per_conn.set_ylabel("USD per connection")
ax_inv_per_conn.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
offgrid_names = ["SA_Diesel", "SA_PV", "MG_Diesel", "MG_PV", "MG_Wind", "MG_Hydro"]

grid_offgrid_df = pd.DataFrame(index=population_mix_df.index)
grid_offgrid_df["Grid"] = population_mix_df["Grid"]
grid_offgrid_df["Off-grid"] = population_mix_df[offgrid_names].sum(axis=1)

fig_grid_offgrid, ax_grid_offgrid = plt.subplots(figsize=(10, 6))
grid_offgrid_df.div(grid_offgrid_df.sum(axis=1), axis=0).mul(100).plot(kind="bar", stacked=True, ax=ax_grid_offgrid)
ax_grid_offgrid.set_title(f"Grid vs off-grid population share ({year})")
ax_grid_offgrid.set_ylabel("Share of electrified population (%)")
ax_grid_offgrid.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 8c. Maps: baseline, climate exclusion, and reassignment

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
clim_col = f"FinalElecCode{year}_{CLIMATE_SCENARIO}"

tech_labels = {
    1: "Grid",
    2: "SA_Diesel",
    3: "SA_PV",
    4: "MG_Diesel",
    5: "MG_PV",
    6: "MG_Wind",
    7: "MG_Hydro"
}

tech_colors = {
    0: "#D3D3D3",
    1: "#73B2FF",
    2: "#EDD100",
    3: "#EDA800",
    4: "#1F6600",
    5: "#98E600",
    6: "#70A800",
    7: "#1FA800"
}

def plot_tech_map(df, code_col, title):
    fig, ax = plt.subplots(figsize=(10, 10))
    plot_codes = df[code_col].fillna(0)

    # Count how many points each code has
    code_counts = {
        code: int((plot_codes == code).sum())
        for code in tech_colors.keys()
    }

    # Plot most common first, least common last
    plot_order = sorted(
        [code for code in tech_colors.keys() if code_counts[code] > 0],
        key=lambda code: code_counts[code],
        reverse=True
    )

    for code in plot_order:
        mask = plot_codes == code
        label = "No Tech" if code == 0 else tech_labels[code]

        ax.scatter(
            df.loc[mask, SET_X_DEG],
            df.loc[mask, SET_Y_DEG],
            s=2,
            c=tech_colors[code],
            linewidths=0,
            alpha=1,
            label=label,
            zorder=2 + (len(plot_order) - plot_order.index(code))
        )

    # Keep legend in technology-code order, not plotting order
    legend_order = [code for code in sorted(tech_colors.keys()) if code_counts[code] > 0]

    legend_handles = [
        mlines.Line2D(
            [], [], color=tech_colors[code], marker='o', linestyle='None',
            markersize=8,
            label=("No Tech" if code == 0 else tech_labels[code])
        )
        for code in legend_order
    ]

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.legend(handles=legend_handles, loc="best")
    plt.tight_layout()
    return fig, ax

# 1. Baseline map
fig_baseline, ax_baseline = plot_tech_map(
    onsseter.df,
    base_col,
    f"Baseline electrification map {year}"
)
plt.show()

# 2. Climate exclusion map
fig_climate, ax_climate = plot_tech_map(
    onsseter.df,
    clim_col,
    f"Climate-exclusion electrification map {year}"
)
plt.show()

# 3. Reassignment map
fig_reassign, ax_reassign = plt.subplots(figsize=(10, 10))

unchanged_mask = ~onsseter.df["reassigned_due_to_exclusion"]
changed_mask = onsseter.df["reassigned_due_to_exclusion"]

ax_reassign.scatter(
    onsseter.df.loc[unchanged_mask, SET_X_DEG],
    onsseter.df.loc[unchanged_mask, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged",
    zorder=1
)

ax_reassign.scatter(
    onsseter.df.loc[changed_mask, SET_X_DEG],
    onsseter.df.loc[changed_mask, SET_Y_DEG],
    s=8,
    c="red",
    linewidths=0,
    label="Reassigned due to exclusion",
    zorder=2
)

ax_reassign.set_xlabel("Longitude")
ax_reassign.set_ylabel("Latitude")
ax_reassign.set_title(f"Settlements reassigned due to climate exclusion {year}")
ax_reassign.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# Maps: climate exclusion scenario (only changed)

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
clim_col = f"FinalElecCode{year}_{CLIMATE_SCENARIO}"

# 1. Climate-exclusion technology map
fig_climate, ax_climate = plot_tech_map(
    onsseter.df,
    clim_col,
    f"Climate-exclusion electrification map {year}"
)
plt.show()

# 2. Changed settlements map: unchanged in grey, changed in the colour they changed to
fig_reassign, ax_reassign = plt.subplots(figsize=(10, 10))

unchanged_mask = ~onsseter.df["reassigned_due_to_exclusion"]
changed_mask = onsseter.df["reassigned_due_to_exclusion"]

# Plot unchanged settlements in grey
ax_reassign.scatter(
    onsseter.df.loc[unchanged_mask, SET_X_DEG],
    onsseter.df.loc[unchanged_mask, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

# Plot changed settlements in the colour of the new technology
changed_codes = onsseter.df.loc[changed_mask, clim_col].fillna(0)
changed_counts = changed_codes.value_counts()

for code in changed_counts.sort_values(ascending=False).index:
    mask = changed_mask & (onsseter.df[clim_col].fillna(0) == code)

    if mask.sum() == 0:
        continue

    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))

    ax_reassign.scatter(
        onsseter.df.loc[mask, SET_X_DEG],
        onsseter.df.loc[mask, SET_Y_DEG],
        s=10,
        c=tech_colors.get(code, "black"),
        linewidths=0,
        label=label
    )

# Legend
legend_handles = [
    mlines.Line2D([], [], color="lightgrey", marker='o', linestyle='None',
                  markersize=6, label="Unchanged")
]

for code in sorted(changed_counts.index):
    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))
    legend_handles.append(
        mlines.Line2D([], [], color=tech_colors.get(code, "black"),
                      marker='o', linestyle='None', markersize=6, label=label)
    )

ax_reassign.set_xlabel("Longitude")
ax_reassign.set_ylabel("Latitude")
ax_reassign.set_title(f"Settlements changed by climate-exclusion scenario {year} (coloured by new technology)")
ax_reassign.legend(handles=legend_handles, loc="best")
plt.tight_layout()
plt.show()

## 8d. Hazard exposure maps

In [ ]:
# 8d. Hazard exposure maps

from matplotlib import pyplot as plt

year = end_year

hazard_plot_settings = {
    "flood": {
        "value_col": "hazard_flood",
        "exposure_col": "exposed_flood",
        "title_value": f"Flood hazard severity {year}",
        "title_exposure": f"Flood exposure {year}"
    },
    "fire": {
        "value_col": "hazard_fire",
        "exposure_col": "exposed_fire",
        "title_value": f"Fire hazard severity {year}",
        "title_exposure": f"Fire exposure {year}"
    },
    "heat": {
        "value_col": "hazard_heat",
        "exposure_col": "exposed_heat",
        "title_value": f"Heat hazard severity {year}",
        "title_exposure": f"Heat exposure {year}"
    },
    "cold": {
        "value_col": "hazard_cold",
        "exposure_col": "exposed_cold",
        "title_value": f"Cold hazard severity {year}",
        "title_exposure": f"Cold exposure {year}"
    },
    "high_precip": {
        "value_col": "hazard_high_precip",
        "exposure_col": "exposed_high_precip",
        "title_value": f"High precipitation severity {year}",
        "title_exposure": f"High precipitation exposure {year}"
    },
    "drought": {
        "value_col": "hazard_drought_spi3",
        "exposure_col": "exposed_drought",
        "title_value": f"Drought SPI-3 severity {year}",
        "title_exposure": f"Drought exposure {year}"
    },
    "landslide": {
        "value_col": "hazard_landslide",
        "exposure_col": "exposed_landslide",
        "title_value": f"Landslide hazard severity {year}",
        "title_exposure": f"Landslide exposure {year}"
    },
    "wind": {
        "value_col": "hazard_wind",
        "exposure_col": "exposed_wind",
        "title_value": f"Wind hazard severity {year}",
        "title_exposure": f"Wind exposure {year}"
    }
}

hazard_figures = {}

for hazard, settings in hazard_plot_settings.items():
    if hazard not in ACTIVE_HAZARDS:
        continue
    value_col = settings["value_col"]
    exposure_col = settings["exposure_col"]

    if value_col not in onsseter.df.columns or exposure_col not in onsseter.df.columns:
        print(f"[WARN] Missing columns for {hazard}, skipping maps.")
        continue

    # 1. Severity map
    plot_df = onsseter.df[[SET_X_DEG, SET_Y_DEG, value_col]].copy()
    plot_df = plot_df.sort_values(by=value_col, ascending=True)
    
    fig_val, ax_val = plt.subplots(figsize=(10, 10))
    sc = ax_val.scatter(
        plot_df[SET_X_DEG],
        plot_df[SET_Y_DEG],
        c=plot_df[value_col],
        s=3,
        linewidths=0
    )
    ax_val.set_xlabel("Longitude")
    ax_val.set_ylabel("Latitude")
    ax_val.set_title(settings["title_value"])
    plt.colorbar(sc, ax=ax_val, label=value_col)
    plt.tight_layout()
    plt.show()

    # 2. Exposure map
    fig_exp, ax_exp = plt.subplots(figsize=(10, 10))

    not_exposed = ~onsseter.df[exposure_col].fillna(False)
    exposed = onsseter.df[exposure_col].fillna(False)

    ax_exp.scatter(
        onsseter.df.loc[not_exposed, SET_X_DEG],
        onsseter.df.loc[not_exposed, SET_Y_DEG],
        s=2,
        c="lightgrey",
        linewidths=0,
        label="Not exposed"
    )

    ax_exp.scatter(
        onsseter.df.loc[exposed, SET_X_DEG],
        onsseter.df.loc[exposed, SET_Y_DEG],
        s=3,
        c="red",
        linewidths=0,
        label="Exposed"
    )

    ax_exp.set_xlabel("Longitude")
    ax_exp.set_ylabel("Latitude")
    ax_exp.set_title(settings["title_exposure"])
    ax_exp.legend(loc="best")
    plt.tight_layout()
    plt.show()

    hazard_figures[f"{hazard}_severity"] = fig_val
    hazard_figures[f"{hazard}_exposure"] = fig_exp

## 8e. Maps: climate-cost scenario

In [ ]:
from matplotlib import pyplot as plt

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_col = f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost technology map
fig_cost, ax_cost = plot_tech_map(
    onsseter.df,
    cost_col,
    f"Climate-cost electrification map {year}"
)
plt.show()

# 2. Changed settlements map: baseline vs climate cost
fig_cost_change, ax_cost_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost = ~onsseter.df["tech_changed_cost"]
changed_mask_cost = onsseter.df["tech_changed_cost"]

ax_cost_change.scatter(
    onsseter.df.loc[unchanged_mask_cost, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

ax_cost_change.scatter(
    onsseter.df.loc[changed_mask_cost, SET_X_DEG],
    onsseter.df.loc[changed_mask_cost, SET_Y_DEG],
    s=10,
    c="blue",
    linewidths=0,
    label="Changed by climate cost"
)

ax_cost_change.set_xlabel("Longitude")
ax_cost_change.set_ylabel("Latitude")
ax_cost_change.set_title(f"Settlements changed by climate-cost scenario {year}")
ax_cost_change.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# 8e. Maps: climate-cost scenario (only changed)
# =========================================================

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_col = f"FinalElecCode{year}_{COST_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost technology map
fig_cost, ax_cost = plot_tech_map(
    onsseter.df,
    cost_col,
    f"Climate-cost electrification map {year}"
)
plt.show()

# 2. Changed settlements map: unchanged in grey, changed in the colour they changed to
fig_cost_change, ax_cost_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost = ~onsseter.df["tech_changed_cost"]
changed_mask_cost = onsseter.df["tech_changed_cost"]

# Plot unchanged settlements in grey
ax_cost_change.scatter(
    onsseter.df.loc[unchanged_mask_cost, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

# Plot changed settlements in the colour of the new technology
changed_codes = onsseter.df.loc[changed_mask_cost, cost_col].fillna(0)

# plot rarer technologies on top
changed_counts = changed_codes.value_counts()

for code in changed_counts.sort_values(ascending=False).index:
    mask = changed_mask_cost & (onsseter.df[cost_col].fillna(0) == code)

    if mask.sum() == 0:
        continue

    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))

    ax_cost_change.scatter(
        onsseter.df.loc[mask, SET_X_DEG],
        onsseter.df.loc[mask, SET_Y_DEG],
        s=10,
        c=tech_colors.get(code, "black"),
        linewidths=0,
        label=label
    )

# Legend
legend_handles = [
    mlines.Line2D([], [], color="lightgrey", marker='o', linestyle='None',
                  markersize=6, label="Unchanged")
]

for code in sorted(changed_counts.index):
    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))
    legend_handles.append(
        mlines.Line2D([], [], color=tech_colors.get(code, "black"),
                      marker='o', linestyle='None', markersize=6, label=label)
    )

ax_cost_change.set_xlabel("Longitude")
ax_cost_change.set_ylabel("Latitude")
ax_cost_change.set_title(f"Settlements changed by climate-cost scenario {year} (coloured by new technology)")
ax_cost_change.legend(handles=legend_handles, loc="best")
plt.tight_layout()
plt.show()

## Maps: climate-cost-v2 scenario

In [ ]:
# =========================================================
# Maps: climate-cost-v2 scenario
# =========================================================

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v2_col = f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost-v2 technology map
fig_cost_v2, ax_cost_v2 = plot_tech_map(
    onsseter.df,
    cost_v2_col,
    f"Climate-cost-v2 electrification map {year}"
)
plt.show()

# 2. Changed settlements map: baseline vs climate cost v2
fig_cost_v2_change, ax_cost_v2_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost_v2 = ~onsseter.df["tech_changed_cost_v2"]
changed_mask_cost_v2 = onsseter.df["tech_changed_cost_v2"]

ax_cost_v2_change.scatter(
    onsseter.df.loc[unchanged_mask_cost_v2, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost_v2, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

ax_cost_v2_change.scatter(
    onsseter.df.loc[changed_mask_cost_v2, SET_X_DEG],
    onsseter.df.loc[changed_mask_cost_v2, SET_Y_DEG],
    s=10,
    c="green",
    linewidths=0,
    label="Changed by climate cost v2"
)

ax_cost_v2_change.set_xlabel("Longitude")
ax_cost_v2_change.set_ylabel("Latitude")
ax_cost_v2_change.set_title(f"Settlements changed by climate-cost-v2 scenario {year}")
ax_cost_v2_change.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Maps: climate-cost-v2 scenario (only changed)
# =========================================================

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v2_col = f"FinalElecCode{year}_{CAPEX_OPEX_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost-v2 technology map
fig_cost_v2, ax_cost_v2 = plot_tech_map(
    onsseter.df,
    cost_v2_col,
    f"Climate-cost-v2 electrification map {year}"
)
plt.show()

# 2. Changed settlements map: unchanged in grey, changed in the colour they changed to
fig_cost_v2_change, ax_cost_v2_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost_v2 = ~onsseter.df["tech_changed_cost_v2"]
changed_mask_cost_v2 = onsseter.df["tech_changed_cost_v2"]

# Plot unchanged settlements in grey
ax_cost_v2_change.scatter(
    onsseter.df.loc[unchanged_mask_cost_v2, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost_v2, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

# Plot changed settlements in the colour of the new technology
changed_codes = onsseter.df.loc[changed_mask_cost_v2, cost_v2_col].fillna(0)
changed_counts = changed_codes.value_counts()

for code in changed_counts.sort_values(ascending=False).index:
    mask = changed_mask_cost_v2 & (onsseter.df[cost_v2_col].fillna(0) == code)

    if mask.sum() == 0:
        continue

    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))

    ax_cost_v2_change.scatter(
        onsseter.df.loc[mask, SET_X_DEG],
        onsseter.df.loc[mask, SET_Y_DEG],
        s=10,
        c=tech_colors.get(code, "black"),
        linewidths=0,
        label=label
    )

# Legend
legend_handles = [
    mlines.Line2D([], [], color="lightgrey", marker='o', linestyle='None',
                  markersize=6, label="Unchanged")
]

for code in sorted(changed_counts.index):
    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))
    legend_handles.append(
        mlines.Line2D([], [], color=tech_colors.get(code, "black"),
                      marker='o', linestyle='None', markersize=6, label=label)
    )

ax_cost_v2_change.set_xlabel("Longitude")
ax_cost_v2_change.set_ylabel("Latitude")
ax_cost_v2_change.set_title(f"Settlements changed by climate-cost-v2 scenario {year} (coloured by new technology)")
ax_cost_v2_change.legend(handles=legend_handles, loc="best")
plt.tight_layout()
plt.show()

## Maps: climate-cost-v3 scenario

In [ ]:
# =========================================================
# Maps: climate-cost-v3 scenario
# =========================================================

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v3_col = f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost-v3 technology map
fig_cost_v3, ax_cost_v3 = plot_tech_map(
    onsseter.df,
    cost_v3_col,
    f"Climate-cost-v3 electrification map {year}"
)
plt.show()

# 2. Changed settlements map: baseline vs climate cost v3
fig_cost_v3_change, ax_cost_v3_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost_v3 = ~onsseter.df["tech_changed_cost_v3"]
changed_mask_cost_v3 = onsseter.df["tech_changed_cost_v3"]

ax_cost_v3_change.scatter(
    onsseter.df.loc[unchanged_mask_cost_v3, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost_v3, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

ax_cost_v3_change.scatter(
    onsseter.df.loc[changed_mask_cost_v3, SET_X_DEG],
    onsseter.df.loc[changed_mask_cost_v3, SET_Y_DEG],
    s=10,
    c="purple",
    linewidths=0,
    label="Changed by climate cost v3"
)

ax_cost_v3_change.set_xlabel("Longitude")
ax_cost_v3_change.set_ylabel("Latitude")
ax_cost_v3_change.set_title(f"Settlements changed by climate-cost-v3 scenario {year}")
ax_cost_v3_change.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# Maps: climate-cost-v3 scenario (only changed)
# =========================================================

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
cost_v3_col = f"FinalElecCode{year}_{LIFECYCLE_ADJUSTMENT_SCENARIO}"

# 1. Climate-cost-v3 technology map
fig_cost_v3, ax_cost_v3 = plot_tech_map(
    onsseter.df,
    cost_v3_col,
    f"Climate-cost-v3 electrification map {year}"
)
plt.show()

# 2. Changed settlements map: unchanged in grey, changed in the colour they changed to
fig_cost_v3_change, ax_cost_v3_change = plt.subplots(figsize=(10, 10))

unchanged_mask_cost_v3 = ~onsseter.df["tech_changed_cost_v3"]
changed_mask_cost_v3 = onsseter.df["tech_changed_cost_v3"]

# Plot unchanged settlements in grey
ax_cost_v3_change.scatter(
    onsseter.df.loc[unchanged_mask_cost_v3, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_cost_v3, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

# Plot changed settlements in the colour of the new technology
changed_codes = onsseter.df.loc[changed_mask_cost_v3, cost_v3_col].fillna(0)
changed_counts = changed_codes.value_counts()

for code in changed_counts.sort_values(ascending=False).index:
    mask = changed_mask_cost_v3 & (onsseter.df[cost_v3_col].fillna(0) == code)

    if mask.sum() == 0:
        continue

    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))

    ax_cost_v3_change.scatter(
        onsseter.df.loc[mask, SET_X_DEG],
        onsseter.df.loc[mask, SET_Y_DEG],
        s=10,
        c=tech_colors.get(code, "black"),
        linewidths=0,
        label=label
    )

# Legend
legend_handles = [
    mlines.Line2D([], [], color="lightgrey", marker='o', linestyle='None',
                  markersize=6, label="Unchanged")
]

for code in sorted(changed_counts.index):
    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))
    legend_handles.append(
        mlines.Line2D([], [], color=tech_colors.get(code, "black"),
                      marker='o', linestyle='None', markersize=6, label=label)
    )

ax_cost_v3_change.set_xlabel("Longitude")
ax_cost_v3_change.set_ylabel("Latitude")
ax_cost_v3_change.set_title(f"Settlements changed by climate-cost-v3 scenario {year} (coloured by new technology)")
ax_cost_v3_change.legend(handles=legend_handles, loc="best")
plt.tight_layout()
plt.show()

## Maps: climate-performance scenario

In [ ]:
year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
perf_col = f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"

# 1. Climate-performance technology map
fig_performance, ax_performance = plot_tech_map(
    onsseter.df,
    perf_col,
    f"Climate-performance electrification map {year}"
)
plt.show()

# 2. Changed settlements map: baseline vs climate performance
fig_performance_change, ax_performance_change = plt.subplots(figsize=(10, 10))

unchanged_mask_perf = ~onsseter.df["tech_changed_performance"]
changed_mask_perf = onsseter.df["tech_changed_performance"]

ax_performance_change.scatter(
    onsseter.df.loc[unchanged_mask_perf, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_perf, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

ax_performance_change.scatter(
    onsseter.df.loc[changed_mask_perf, SET_X_DEG],
    onsseter.df.loc[changed_mask_perf, SET_Y_DEG],
    s=10,
    c="orange",
    linewidths=0,
    label="Changed by climate performance"
)

ax_performance_change.set_xlabel("Longitude")
ax_performance_change.set_ylabel("Latitude")
ax_performance_change.set_title(f"Settlements changed by climate-performance scenario {year}")
ax_performance_change.legend(loc="best")
plt.tight_layout()
plt.show()

In [ ]:
# Maps: climate-performance scenario (only changed)

from matplotlib import pyplot as plt
import matplotlib.lines as mlines

year = end_year

base_col = f"FinalElecCode{year}_{BASELINE_SCENARIO}"
perf_col = f"FinalElecCode{year}_{PERFORMANCE_ADJUSTMENT_SCENARIO}"

# 1. Climate-performance technology map
fig_performance, ax_performance = plot_tech_map(
    onsseter.df,
    perf_col,
    f"Climate-performance electrification map {year}"
)
plt.show()

# 2. Changed settlements map: unchanged in grey, changed in the colour they changed to
fig_performance_change, ax_performance_change = plt.subplots(figsize=(10, 10))

unchanged_mask_perf = ~onsseter.df["tech_changed_performance"]
changed_mask_perf = onsseter.df["tech_changed_performance"]

# Plot unchanged settlements in grey
ax_performance_change.scatter(
    onsseter.df.loc[unchanged_mask_perf, SET_X_DEG],
    onsseter.df.loc[unchanged_mask_perf, SET_Y_DEG],
    s=2,
    c="lightgrey",
    linewidths=0,
    label="Unchanged"
)

# Plot changed settlements in the colour of the new tech
changed_codes = onsseter.df.loc[changed_mask_perf, perf_col].fillna(0)
changed_counts = changed_codes.value_counts()

for code in changed_counts.sort_values(ascending=False).index:
    mask = changed_mask_perf & (onsseter.df[perf_col].fillna(0) == code)

    if mask.sum() == 0:
        continue

    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))

    ax_performance_change.scatter(
        onsseter.df.loc[mask, SET_X_DEG],
        onsseter.df.loc[mask, SET_Y_DEG],
        s=10,
        c=tech_colors.get(code, "black"),
        linewidths=0,
        label=label
    )

# Legend
legend_handles = [
    mlines.Line2D([], [], color="lightgrey", marker='o', linestyle='None',
                  markersize=6, label="Unchanged")
]

for code in sorted(changed_counts.index):
    label = "No Tech" if code == 0 else tech_labels.get(code, str(code))
    legend_handles.append(
        mlines.Line2D([], [], color=tech_colors.get(code, "black"),
                      marker='o', linestyle='None', markersize=6, label=label)
    )

ax_performance_change.set_xlabel("Longitude")
ax_performance_change.set_ylabel("Latitude")
ax_performance_change.set_title(f"Settlements changed by climate-performance scenario {year} (coloured by new technology)")
ax_performance_change.legend(handles=legend_handles, loc="best")
plt.tight_layout()
plt.show()

## 9. Exporting results

This code generates csv files:
 - one containing all the results for the scenario created
 - one containing the summary for the scenario created
 - one containing some of the key input variables of the scenario

Before we proceed, please write the scenario_name in the first cell below. then move on to the next cell and run it to browse to the directory where you want to save your results. Sample file shall be located at .\ gep-onsset\sample_output. 

**Note that if you do not change the scenario name, the previous output files will be overwritten**

In [ ]:
scenario_name = "acclimate"

In [ ]:
list1 = [('Start_year',start_year,'','',''), 
         ('End_year',end_year,'','',''),
         ('End year electrification rate target',electrification_rate_target,'','',''),
         ('Intermediate target year', intermediate_year,'','',''),
         ('Intermediate electrification rate target', intermediate_electrification_target,'','',''),
         ('Urban target tier', urban_target_tier, '', '', ''),
         ('Rural target tier', rural_target_tier, '', '', ''),
         ('Prioritization', prioritization,'','','1 = baseline, 2 = intensification'),
         ('Auto intensification distance', auto_intensification, '', '', 'Buffer distance (km) for automatic intensification if choosing prioritization 1'),
         ('discount_rate',discount_rate,'','',''),
         ('pop_start_year',pop_start_year,'','','the population in the base year (e.g. 2016)'),
         ('pop_end_year',end_year_pop,'','','the projected population in the end year (e.g. 2030)'),
         ('urban_ratio_start_year',urban_ratio_start_year,'','','the urban population population ratio in the base year (e.g. 2016)'),
         ('urban_ratio_end_year',urban_ratio_end_year,'','','the urban population population ratio in the end year (e.g. 2030)'),
         ('num_people_per_hh_urban',num_people_per_hh_urban,'','','the number of people per household expected in the end year (e.g. 2030)'),
         ('num_people_per_hh_rural',num_people_per_hh_rural,'','','the number of people per household expected in the end year (e.g. 2030)'),
         ('elec_ratio_start_year',elec_ratio_start_year,'','','the electrification rate in the base year (e.g. 2016)'),
         ('urban_elec_ratio',urban_elec_ratio,'','','urban electrification rate in the base year (e.g. 2016)'),
         ('rural_elec_ratio',rural_elec_ratio,'','','rural electrification rate in the base year (e.g. 2016)'),
         ('grid_generation_cost',grid_generation_cost,'','','This is the grid cost electricity USD/kWh as expected in the end year of the analysis'),
         ('grid_power_plants_capital_cost',grid_power_plants_capital_cost,'','','The cost in USD/kW to for capacity upgrades of the grid-connected power plants'),
         ('grid_losses',grid_losses,'','','The fraction of electricity lost in transmission and distribution (percentage)'),
         ('base_to_peak',base_to_peak,'','','The ratio of base grid demand to peak demand (percentage)'),
         ('existing_grid_cost_ratio',existing_grid_cost_ratio,'','','The additional cost per round of electrification (percentage)'),
         ('diesel_price',diesel_price,'','','This is the diesel price in USD/liter as expected in the end year of the analysis'),
         ('sa_diesel_capital_cost',sa_diesel_capital_cost,'','','Stand-alone Diesel capital cost (USD/kW) as expected in the years of the analysis'),
         ('mg_diesel_capital_cost',mg_diesel_capital_cost,'','','Mini-grid Diesel capital cost (USD/kW) as expected in the years of the analysis'),
         ('mg_pv_capital_cost',mg_pv_capital_cost,'','','Mini-grid PV capital cost (USD/kW) as expected in the years of the analysis'),
         ('mg_wind_capital_cost',mg_wind_capital_cost,'','','Mini-grid Wind capital cost (USD/kW) as expected in the years of the analysis'),
         ('mg_hydro_capital_cost',mg_hydro_capital_cost,'','','Mini-grid Hydro capital cost (USD/kW) as expected in the years of the analysis'),
         ('sa_pv_capital_cost_1',sa_pv_capital_cost_1,'','','Stand-alone PV capital cost (USD/kW) for household systems under 20 W'),
         ('sa_pv_capital_cost_2',sa_pv_capital_cost_2,'','','Stand-alone PV capital cost (USD/kW) for household systems between 21-50 W'),
         ('sa_pv_capital_cost_3',sa_pv_capital_cost_3,'','','Stand-alone PV capital cost (USD/kW) for household systems between 51-100 W'),
         ('sa_pv_capital_cost_4',sa_pv_capital_cost_4,'','','Stand-alone PV capital cost (USD/kW) for household systems between 101-200 W'),
         ('sa_pv_capital_cost_5',sa_pv_capital_cost_5,'','','Stand-alone PV capital cost (USD/kW) for household systems over 200 W'),
         ('mv_line_cost',mv_line_cost,'','','Cost of MV lines in USD/km'),
         ('lv_line_cost',lv_line_cost,'','','Cost of LV lines in USD/km'),
         ('mv_line_capacity',mv_line_capacity,'','','Capacity of MV lines in kW/line'),
         ('lv_line_capacity',lv_line_capacity,'','','Capacity of LV lines in kW/line'),
         ('lv_line_max_length',lv_line_max_length,'','','Maximum length of LV lines (km)'),
         ('hv_line_cost',hv_line_cost,'','','Cost of HV lines in USD/km'),
         ('mv_line_max_length',mv_line_max_length,'','','Maximum length of MV lines (km)'),
         ('hv_lv_transformer_cost',hv_lv_transformer_cost,'','','Cost of HV/MV transformer (USD/unit)'),
         ('mv_increase_rate',mv_increase_rate,'','','percentage'),
         ('max_grid_extension_dist',max_mv_line_dist,'','','Maximum distance that the grid may be extended by means of MV lines'),
         ('annual_new_grid_connections_limit', annual_new_grid_connections_limit,'','','This is the maximum amount of new households that can be connected to the grid in one year (thousands)'),
         ('grid_capacity_limit',annual_grid_cap_gen_limit,'','','This is the maximum generation capacity that can be added to the grid in one year (MW)'),
         ('GIS data: Administrative boundaries','','','','Delineates the boundaries of the analysis.'),
         ('GIS data: DEM','','','','Filled DEM (elevation) maps are use in a number of processes in the analysis (Energy potentials, restriction zones, grid extension suitability map etc.).'),
         ('GIS data: Hydropower','','','','Points showing potential mini/small hydropower potential.  Provides power availability in each identified point.'),
         ('GIS data: Land Cover','','','','Land cover maps are use in a number of processes in the analysis (Energy potentials, restriction zones, grid extension suitability map etc.).'),
         ('GIS data: Night-time Lights','','','','Dataset used to,identify and spatially calibrate the currently electrified/non-electrified population.'),
         ('GIS data: Population','','','','Spatial identification and quantification of the current (base year) population. This dataset sets the basis of the ONSSET analysis as it is directly connected with the electricity demand and the assignment of energy access goals'),
         ('GIS data: Roads','','','','Current road infrastructure is used in order to specify grid extension suitability.'),
         ('GIS data: Solar GHI','','','','Provide information about the Global Horizontal Irradiation (kWh/m2/year) over an area. This is later used to identify the availability/suitability of Photovoltaic systems.'),
         ('GIS data: Substations','','','','Current Substation infrastructure is used in order to specify grid extension suitability.'),
         ('GIS data: Existing grid','','','','Current grid network'),
         ('GIS data: Planned grid','','','','Planned/committed grid network extensions'),
         ('GIS data: Travel-time','','','','Visualizes spatially the travel time required to reach from any individual cell to the closest town with population more than 50,000 people.'),
         ('GIS data: Wind velocity','','','','Provide information about the wind velocity (m/sec) over an area. This is later used to identify the availability/suitability of wind power (using Capacity factors).'),
        ]
labels = ['Variable','Value', 'Source', 'Comments', 'Description']
df_variables = pd.DataFrame.from_records(list1, columns=labels)

In [ ]:
messagebox.showinfo('OnSSET', 'Browse to the folder where you want to save the outputs')

output_dir = filedialog.askdirectory()

# Output paths

# Core csv outputs
output_dir_variables = os.path.join(output_dir, f'{scenario_name}_Variables.csv')
output_dir_results = os.path.join(output_dir, f'{scenario_name}_Results.csv')
output_dir_summaries = os.path.join(output_dir, f'{scenario_name}_Summaries.csv')

# Scenario comparison csv outputs
output_dir_change_matrix_exclusion = os.path.join(output_dir, f'{scenario_name}_ChangeMatrix_ClimateExclusion_{end_year}.csv')
output_dir_change_matrix_cost = os.path.join(output_dir, f'{scenario_name}_ChangeMatrix_ClimateCost_{end_year}.csv')
output_dir_change_matrix_cost_v2 = os.path.join(output_dir, f'{scenario_name}_ChangeMatrix_ClimateCostV2_{end_year}.csv')
output_dir_change_matrix_cost_v3 = os.path.join(output_dir, f'{scenario_name}_ChangeMatrix_ClimateCostV3_{end_year}.csv')
output_dir_change_matrix_performance = os.path.join(output_dir, f'{scenario_name}_ChangeMatrix_ClimatePerformance_{end_year}.csv')

output_dir_comparison_summary = os.path.join(output_dir, f'{scenario_name}_ScenarioComparison_{end_year}.csv')
output_dir_settlement_mix_csv = os.path.join(output_dir, f'{scenario_name}_SettlementsByTechnology_{end_year}.csv')
output_dir_settlement_share_csv = os.path.join(output_dir, f'{scenario_name}_SettlementShareByTechnology_{end_year}.csv')
output_dir_population_mix_csv = os.path.join(output_dir, f'{scenario_name}_PopulationByTechnology_{end_year}.csv')
output_dir_shift_population_csv = os.path.join(output_dir, f'{scenario_name}_PopulationShiftedFromBaseline_{end_year}.csv')
output_dir_changed_summary_csv = os.path.join(output_dir, f'{scenario_name}_ChangedSettlementsSummary_{end_year}.csv')

# Delta csv outputs
output_dir_comparison_delta_csv = os.path.join(output_dir, f'{scenario_name}_TechnologyCountDifferencesFromBaseline_{end_year}.csv')
output_dir_population_delta_csv = os.path.join(output_dir, f'{scenario_name}_PopulationDifferencesFromBaseline_{end_year}.csv')

# Main figure outputs
output_dir_summary_plot = os.path.join(output_dir, f'{scenario_name}_Summary_Barcharts_{end_year}.png')

output_dir_baseline_map = os.path.join(output_dir, f'{scenario_name}_Baseline_Map_{end_year}.png')
output_dir_climate_map = os.path.join(output_dir, f'{scenario_name}_ClimateExclusion_Map_{end_year}.png')
output_dir_reassign_map = os.path.join(output_dir, f'{scenario_name}_ClimateExclusion_Change_Map_{end_year}.png')

output_dir_cost_map = os.path.join(output_dir, f'{scenario_name}_ClimateCost_Map_{end_year}.png')
output_dir_cost_change_map = os.path.join(output_dir, f'{scenario_name}_ClimateCost_Change_Map_{end_year}.png')

output_dir_cost_v2_map = os.path.join(output_dir, f'{scenario_name}_ClimateCostV2_Map_{end_year}.png')
output_dir_cost_v2_change_map = os.path.join(output_dir, f'{scenario_name}_ClimateCostV2_Change_Map_{end_year}.png')

output_dir_cost_v3_map = os.path.join(output_dir, f'{scenario_name}_ClimateCostV3_Map_{end_year}.png')
output_dir_cost_v3_change_map = os.path.join(output_dir, f'{scenario_name}_ClimateCostV3_Change_Map_{end_year}.png')

output_dir_performance_map = os.path.join(output_dir, f'{scenario_name}_ClimatePerformance_Map_{end_year}.png')
output_dir_performance_change_map = os.path.join(output_dir, f'{scenario_name}_ClimatePerformance_Change_Map_{end_year}.png')

# Non-map figure outputs
output_dir_settlement_mix_plot = os.path.join(output_dir, f'{scenario_name}_SettlementsByTechnology_{end_year}.png')
output_dir_settlement_share_plot = os.path.join(output_dir, f'{scenario_name}_SettlementShareByTechnology_{end_year}.png')
output_dir_population_mix_plot = os.path.join(output_dir, f'{scenario_name}_PopulationByTechnology_{end_year}.png')
output_dir_shift_population_plot = os.path.join(output_dir, f'{scenario_name}_PopulationShiftedFromBaseline_{end_year}.png')
output_dir_changed_methods_plot = os.path.join(output_dir, f'{scenario_name}_ChangedSettlementsByMethod_{end_year}.png')
output_dir_combined_delta_plot = os.path.join(output_dir, f'{scenario_name}_CombinedDelta_Settlements_Population_{end_year}.png')

# Heatmaps
output_dir_heatmap_exclusion = os.path.join(output_dir, f'{scenario_name}_Heatmap_Exclusion_{end_year}.png')
output_dir_heatmap_cost = os.path.join(output_dir, f'{scenario_name}_Heatmap_CostV1_{end_year}.png')
output_dir_heatmap_cost_v2 = os.path.join(output_dir, f'{scenario_name}_Heatmap_CostV2_{end_year}.png')
output_dir_heatmap_cost_v3 = os.path.join(output_dir, f'{scenario_name}_Heatmap_CostV3_{end_year}.png')
output_dir_heatmap_performance = os.path.join(output_dir, f'{scenario_name}_Heatmap_Performance_{end_year}.png')

In [ ]:
# Save csv outputs

# Main outputs
onsseter.df.to_csv(output_dir_results, index=False)
summary_table.to_csv(output_dir_summaries, index=True)
df_variables.to_csv(output_dir_variables, index=False)

# Change matrices
if 'change_matrix' in globals():
    change_matrix.to_csv(output_dir_change_matrix_exclusion)

if 'change_matrix_cost' in globals():
    change_matrix_cost.to_csv(output_dir_change_matrix_cost)

if 'change_matrix_cost_v2' in globals():
    change_matrix_cost_v2.to_csv(output_dir_change_matrix_cost_v2)

if 'change_matrix_cost_v3' in globals():
    change_matrix_cost_v3.to_csv(output_dir_change_matrix_cost_v3)

if 'change_matrix_performance' in globals():
    change_matrix_performance.to_csv(output_dir_change_matrix_performance)

# Scenario comparison tables
if 'comparison_summary' in globals():
    comparison_summary.to_csv(output_dir_comparison_summary)

if 'settlement_mix_df' in globals():
    settlement_mix_df.to_csv(output_dir_settlement_mix_csv)

if 'settlement_mix_share_df' in globals():
    settlement_mix_share_df.to_csv(output_dir_settlement_share_csv)

if 'population_mix_df' in globals():
    population_mix_df.to_csv(output_dir_population_mix_csv)

if 'shift_summary_df' in globals():
    shift_summary_df.to_csv(output_dir_shift_population_csv, index=False)

if 'changed_summary' in globals():
    changed_summary.to_csv(output_dir_changed_summary_csv, index=False)

# Delta tables
if 'comparison_summary_delta' in globals():
    comparison_summary_delta.to_csv(output_dir_comparison_delta_csv)

if 'population_mix_delta_df' in globals():
    population_mix_delta_df.to_csv(output_dir_population_delta_csv)

# Save figure outputs

# Main summary plot
if 'f' in globals():
    f.savefig(output_dir_summary_plot, dpi=300, bbox_inches="tight", facecolor="white")

# Main maps
if 'fig_baseline' in globals():
    fig_baseline.savefig(output_dir_baseline_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_climate' in globals():
    fig_climate.savefig(output_dir_climate_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_reassign' in globals():
    fig_reassign.savefig(output_dir_reassign_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost' in globals():
    fig_cost.savefig(output_dir_cost_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost_change' in globals():
    fig_cost_change.savefig(output_dir_cost_change_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost_v2' in globals():
    fig_cost_v2.savefig(output_dir_cost_v2_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost_v2_change' in globals():
    fig_cost_v2_change.savefig(output_dir_cost_v2_change_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost_v3' in globals():
    fig_cost_v3.savefig(output_dir_cost_v3_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_cost_v3_change' in globals():
    fig_cost_v3_change.savefig(output_dir_cost_v3_change_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_performance' in globals():
    fig_performance.savefig(output_dir_performance_map, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_performance_change' in globals():
    fig_performance_change.savefig(output_dir_performance_change_map, dpi=300, bbox_inches="tight", facecolor="white")

# Hazard maps
if 'hazard_figures' in globals():
    for hazard in ACTIVE_HAZARDS:
        severity_key = f"{hazard}_severity"
        exposure_key = f"{hazard}_exposure"

        if severity_key in hazard_figures:
            hazard_figures[severity_key].savefig(
                os.path.join(output_dir, f"{scenario_name}_{hazard.capitalize()}_Severity_{end_year}.png"),
                dpi=300,
                bbox_inches="tight",
                facecolor="white"
            )

        if exposure_key in hazard_figures:
            hazard_figures[exposure_key].savefig(
                os.path.join(output_dir, f"{scenario_name}_{hazard.capitalize()}_Exposure_{end_year}.png"),
                dpi=300,
                bbox_inches="tight",
                facecolor="white"
            )

# Non-map figures
if 'fig_settlement_mix' in globals():
    fig_settlement_mix.savefig(output_dir_settlement_mix_plot, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_settlement_share' in globals():
    fig_settlement_share.savefig(output_dir_settlement_share_plot, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_population_mix' in globals():
    fig_population_mix.savefig(output_dir_population_mix_plot, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_shift_population' in globals():
    fig_shift_population.savefig(output_dir_shift_population_plot, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_changed_methods' in globals():
    fig_changed_methods.savefig(output_dir_changed_methods_plot, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_combined_delta' in globals():
    fig_combined_delta.savefig(output_dir_combined_delta_plot, dpi=300, bbox_inches="tight", facecolor="white")

# Heatmaps
if 'fig_heatmap_exclusion' in globals():
    fig_heatmap_exclusion.savefig(output_dir_heatmap_exclusion, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_heatmap_cost' in globals():
    fig_heatmap_cost.savefig(output_dir_heatmap_cost, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_heatmap_cost_v2' in globals():
    fig_heatmap_cost_v2.savefig(output_dir_heatmap_cost_v2, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_heatmap_cost_v3' in globals():
    fig_heatmap_cost_v3.savefig(output_dir_heatmap_cost_v3, dpi=300, bbox_inches="tight", facecolor="white")

if 'fig_heatmap_performance' in globals():
    fig_heatmap_performance.savefig(output_dir_heatmap_performance, dpi=300, bbox_inches="tight", facecolor="white")

print("Outputs saved to:", output_dir)